# scholar_research 驗證指標：範例手冊

搭配 [`2026-09-23_validation-plan.md`](2026-09-23_validation-plan.md)。**每個指標都有一個最小可跑的範例**：先列數學公式，再給範例 input，最後是程式與 output。

- 資料全部是**虛構的 toy 資料**（例如 EEG neurofeedback for MCI 的 9 篇論文、4 題 LitSearch 風格問題），目的是把計算過程講清楚，數字本身沒有意義。
- 需要 LLM 或 Prometheus 的地方，用 **mock judge**（寫死的模擬輸出）代替，但 prompt 的組法和輸出的解析都是真的，可以直接換成真模型。
- 只依賴 `numpy`（`scipy` 只在 L6 用來交叉驗算，沒有也能跑）。

| 章節 | 驗證對象 | 指標 | 對應 plan |
|---|---|---|---|
| L0 | 引擎可重現性 | Jaccard、run-to-run 雜訊底線 | §2 |
| L1 | Step 2 檢索 + Step 3 排序 | Recall@k、MRR、nDCG@10、set recall、同分期望值、ID 對映、損失漏斗、Mode A/B、主題層級 recall | §3 |
| L2 | Step 6 確定性檢查 | key 正規化、integrity 硬閘、EN/ZH 一致性、結構、≥80% coverage | §4 |
| L3 | Step 6 引用品質 | 切句、citable、數字 attestation、r_i、must-cite、Cite-p / Cite-r / F1 | §4.4、§5 |
| L4 | Step 6 內容品質 | Prometheus organization / coverage / relevance | §6 |
| L5 | Step 8 假說 | ResearchAgent 15 項 rubric、Insight Score、P2−P1、BERTScore、idea overlap、連貫性前置檢查、Step 7 校準 | §7 |
| L6 | 評審器可信度 | Cohen's κ、Pearson、Spearman、Kendall τ-b | §8 |
| 總表 | 全部指標 | 分區塊回報（不混成一個平均） | §0 |

**換成真實 session 時**，各個 toy 變數對應到的檔案：

| toy 變數 | 真實來源 |
|---|---|
| `KEYS`, `ACCESS` | `step4_citation_keys.json`、`step5_full_text/{key}.md` frontmatter 的 `access_level` |
| `FULLTEXT` | `step5_full_text/{key}.md` 正文 |
| `SOTA` | `step6_sota.json`（`themes[*].consensus_en …`、`paper_map`） |
| `STEP7_GAPS` | `step7_gaps.json` |
| `STEP8_SPEC` | `step8_hypothesis.json` 的 `spec` |
| `litsearch` 的 `ranked` | `step3_screening_all.json` 依 `composite` 排序 |
| `step2_set` | `step2_raw_papers.json` 的 `papers[*]` |

In [2]:
import itertools
import json
import math
import random
import re
from collections import Counter
from decimal import Decimal

import numpy as np


def show(label, obj):
    """Print a labelled value; dicts/lists as pretty JSON."""
    print(f"── {label}")
    if isinstance(obj, (dict, list)):
        print(json.dumps(obj, ensure_ascii=False, indent=2))
    else:
        print(obj)
    print()


def table(rows, headers):
    """Minimal fixed-width table printer (no pandas needed)."""
    rows = [[str(c) for c in r] for r in rows]
    widths = [max(len(h), *(len(r[i]) for r in rows)) if rows else len(h) for i, h in enumerate(headers)]
    print(" | ".join(h.ljust(w) for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(" | ".join(c.ljust(w) for c, w in zip(r, widths)))
    print()


# ---- 統計工具（公式與驗算見 L6，這裡先定義，後面章節會用到） ----
def rankdata(x):
    """Average ranks (1-based); ties share the mean of their positions."""
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x))
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2 + 1
        i = j + 1
    return ranks


def pearson(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    dx, dy = x - x.mean(), y - y.mean()
    return float((dx * dy).sum() / math.sqrt((dx ** 2).sum() * (dy ** 2).sum()))


def spearman(x, y):
    return pearson(rankdata(x), rankdata(y))


def kendall_tau_b(x, y):
    n_c = n_d = t_x = t_y = 0
    for i, j in itertools.combinations(range(len(x)), 2):
        sx, sy = np.sign(x[i] - x[j]), np.sign(y[i] - y[j])
        if sx == 0 and sy == 0:
            continue
        if sx == 0:
            t_x += 1
        elif sy == 0:
            t_y += 1
        elif sx == sy:
            n_c += 1
        else:
            n_d += 1
    return (n_c - n_d) / math.sqrt((n_c + n_d + t_x) * (n_c + n_d + t_y))


def cohen_kappa(a, b):
    n = len(a)
    cats = sorted(set(a) | set(b))
    p_o = sum(x == y for x, y in zip(a, b)) / n
    p_e = sum((a.count(c) / n) * (b.count(c) / n) for c in cats)
    return (p_o - p_e) / (1 - p_e), p_o, p_e


REPORT = {}  # 各章節把最終指標寫進來，最後一格印總表
print("ready")

ready


---
# L0 引擎可重現性：run-to-run 變異

同一個 topic 用 `--refresh` 重跑 R 次，比較 shortlist 的重疊程度，以及主題數、locked gap 是否一致。

**Jaccard 相似度**（兩次 run 的 shortlist $S_a, S_b$）：

$$J(S_a,S_b)=\frac{|S_a\cap S_b|}{|S_a\cup S_b|}\in[0,1]$$

**平均兩兩 Jaccard**（R 次 run）：

$$\bar J=\frac{2}{R(R-1)}\sum_{a<b}J(S_a,S_b)$$

**雜訊底線**：同一版本跑 R 次，得到某指標（例如 Cite-F1）的標準差 $\sigma_{\text{run}}$。新舊版本的差距 $\Delta$ 若 $|\Delta|\le 2\sigma_{\text{run}}$，就不能說哪個版本比較好。這個 $2\sigma$ 是經驗法則，屬於我的建議，不是來自 material。

In [ ]:
# ---- input：3 次 fresh run 的摘要 ----
runs = {
    "run1": {"shortlist": {"paper_001", "paper_002", "paper_003", "paper_005", "paper_008"}, "n_themes": 4, "locked_gap": "Long-term follow-up of NF effects", "cite_f1": 0.81},
    "run2": {"shortlist": {"paper_001", "paper_002", "paper_004", "paper_005", "paper_008"}, "n_themes": 4, "locked_gap": "Long-term follow-up of NF effects", "cite_f1": 0.78},
    "run3": {"shortlist": {"paper_001", "paper_002", "paper_003", "paper_005", "paper_009", "paper_010"}, "n_themes": 5, "locked_gap": "Responder prediction from baseline EEG", "cite_f1": 0.84},
}


def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 1.0


rows = []
for a, b in itertools.combinations(runs, 2):
    A, B = runs[a]["shortlist"], runs[b]["shortlist"]
    rows.append((a, b, len(A & B), len(A | B), f"{jaccard(A, B):.3f}"))
table(rows, ["run_a", "run_b", "|A∩B|", "|A∪B|", "Jaccard"])

mean_j = float(np.mean([jaccard(runs[a]["shortlist"], runs[b]["shortlist"]) for a, b in itertools.combinations(runs, 2)]))
gap_mode, gap_cnt = Counter(r["locked_gap"] for r in runs.values()).most_common(1)[0]
sigma_run = float(np.std([r["cite_f1"] for r in runs.values()], ddof=1))
delta_version = 0.02  # 假設新版本 Cite-F1 比舊版高 0.02

l0 = {
    "mean_pairwise_jaccard": round(mean_j, 3),
    "n_themes_values": [r["n_themes"] for r in runs.values()],
    "locked_gap_agreement": f"{gap_cnt}/{len(runs)}",
    "cite_f1_sigma_run": round(sigma_run, 4), # 不穩定程度
    "version_delta": delta_version, # 新舊版本差
    "delta_is_meaningful(|Δ|>2σ)": abs(delta_version) > 2 * sigma_run, # 判斷規則 $$|\Delta| > 2\sigma_{\text{run}}$$
}
show("L0 output", l0)
REPORT["L0_reproducibility"] = l0

run_a | run_b | |A∩B| | |A∪B| | Jaccard
------+-------+-------+-------+--------
run1  | run2  | 4     | 6     | 0.667  
run1  | run3  | 4     | 7     | 0.571  
run2  | run3  | 3     | 8     | 0.375  

── L0 output
{
  "mean_pairwise_jaccard": 0.538,
  "n_themes_values": [
    4,
    4,
    5
  ],
  "locked_gap_agreement": "2/3",
  "cite_f1_sigma_run": 0.03,
  "version_delta": 0.02,
  "delta_is_meaningful(|Δ|>2σ)": false
}



---
# L1 Step 2 檢索 + Step 3 排序（LitSearch）

## L1.1 Recall@k、MRR、nDCG@10

設查詢集合為 $Q$，查詢 $q$ 的 gold paper 集合為 $G_q$，系統回傳的排序清單為 $L_q$，$L_q[1..k]$ 是前 k 名。

**Recall@k**：

$$\text{Recall@}k(q)=\frac{|G_q\cap L_q[1..k]|}{|G_q|},\qquad \text{Recall@}k=\frac{1}{|Q|}\sum_{q\in Q}\text{Recall@}k(q)$$

**MRR**（第一篇 gold 出現的名次取倒數；完全沒找到記 0）：

$$\text{RR}(q)=\frac{1}{\min\{i: L_q[i]\in G_q\}},\qquad \text{MRR}=\frac{1}{|Q|}\sum_q \text{RR}(q)$$

**nDCG@10**（二元相關度 $rel_i=\mathbb 1[L_q[i]\in G_q]$）：

$$\text{DCG@}k=\sum_{i=1}^{k}\frac{rel_i}{\log_2(i+1)},\qquad \text{IDCG@}k=\sum_{i=1}^{\min(k,|G_q|)}\frac{1}{\log_2(i+1)},\qquad \text{nDCG@}k=\frac{\text{DCG@}k}{\text{IDCG@}k}$$

**LitSearch 的回報慣例**：specific 題報 R@5 和 R@20，broad 題報 R@20；跨 benchmark 比較用 nDCG@10（論文 Table 8）。MRR 是 Joanna 提議加的，論文沒有報，所以沒有官方 baseline。

**範例 input**：4 題，分別涵蓋 inline / author × broad / specific 四格。問題句取自 LitSearch 論文中的例子；gold 用代號，排序清單是虛構的。`ranked` 代表 Step 3 依 composite 排序後的前 30 名。

In [4]:
def ranked(n, gold_at):
    """Toy ranked list of length n; gold_at = {paper_id: rank (1-based)}."""
    lst = [f"x{i:03d}" for i in range(1, n + 1)]
    for pid, r in gold_at.items():
        lst[r - 1] = pid
    return lst


litsearch = [
    {"qid": "ls_001", "type": "inline", "spec": "specific",
     "question": "Which method involves training additional prompt tokens for every layer during the fine-tuning of language models?",
     "gold": ["PrefixTuning"], "ranked": ranked(30, {"PrefixTuning": 2})},
    {"qid": "ls_002", "type": "inline", "spec": "broad",
     "question": "Is there a pre-trained LM that is trained with reduced sequence length for the first 90% steps?",
     "gold": ["BERT"], "ranked": ranked(30, {"BERT": 7})},
    {"qid": "ls_003", "type": "author", "spec": "specific",
     "question": "Can you find a research paper that uses structured pruning techniques to scale down language models with billions of parameters?",
     "gold": ["ShearedLLaMA", "CoFiPruning"], "ranked": ranked(30, {"ShearedLLaMA": 1, "CoFiPruning": 25})},
    {"qid": "ls_004", "type": "author", "spec": "broad",
     "question": "Is there such a machine learning dataset where for some questions there is no correct answer and the model should abstain?",
     "gold": ["SQuADv2"], "ranked": ranked(30, {})},
]
show("input 範例（第 1 題，ranked 只列前 5）", {**litsearch[0], "ranked": litsearch[0]["ranked"][:5] + ["…"]})

── input 範例（第 1 題，ranked 只列前 5）
{
  "qid": "ls_001",
  "type": "inline",
  "spec": "specific",
  "question": "Which method involves training additional prompt tokens for every layer during the fine-tuning of language models?",
  "gold": [
    "PrefixTuning"
  ],
  "ranked": [
    "x001",
    "PrefixTuning",
    "x003",
    "x004",
    "x005",
    "…"
  ]
}



In [5]:
def recall_at_k(gold, ranked_list, k):
    return len(set(gold) & set(ranked_list[:k])) / len(gold)


def reciprocal_rank(gold, ranked_list):
    for i, pid in enumerate(ranked_list, 1):
        if pid in gold:
            return 1 / i
    return 0.0


def ndcg_at_k(gold, ranked_list, k=10):
    dcg = sum(1 / math.log2(i + 1) for i, pid in enumerate(ranked_list[:k], 1) if pid in gold)
    idcg = sum(1 / math.log2(i + 1) for i in range(1, min(k, len(gold)) + 1))
    return dcg / idcg


rows = []
for q in litsearch:
    q["R@5"] = recall_at_k(q["gold"], q["ranked"], 5)
    q["R@20"] = recall_at_k(q["gold"], q["ranked"], 20)
    q["RR"] = reciprocal_rank(q["gold"], q["ranked"])
    q["nDCG@10"] = ndcg_at_k(q["gold"], q["ranked"], 10)
    gold_ranks = [q["ranked"].index(g) + 1 if g in q["ranked"] else "—" for g in q["gold"]]
    rows.append((q["qid"], q["type"], q["spec"], gold_ranks, f"{q['R@5']:.2f}", f"{q['R@20']:.2f}", f"{q['RR']:.3f}", f"{q['nDCG@10']:.3f}"))
table(rows, ["qid", "type", "spec", "gold 名次", "R@5", "R@20", "RR", "nDCG@10"])

spec = [q for q in litsearch if q["spec"] == "specific"]
broad = [q for q in litsearch if q["spec"] == "broad"]
l1 = {
    "specific_R@5": round(np.mean([q["R@5"] for q in spec]), 3),
    "specific_R@20": round(np.mean([q["R@20"] for q in spec]), 3),
    "broad_R@20": round(np.mean([q["R@20"] for q in broad]), 3),
    "MRR": round(np.mean([q["RR"] for q in litsearch]), 3),
    "nDCG@10": round(np.mean([q["nDCG@10"] for q in litsearch]), 3),
}
show("L1.1 output（依 LitSearch 慣例彙總）", l1)

qid    | type   | spec     | gold 名次 | R@5  | R@20 | RR    | nDCG@10
-------+--------+----------+---------+------+------+-------+--------
ls_001 | inline | specific | [2]     | 1.00 | 1.00 | 0.500 | 0.631  
ls_002 | inline | broad    | [7]     | 0.00 | 1.00 | 0.143 | 0.333  
ls_003 | author | specific | [1, 25] | 0.50 | 0.50 | 1.000 | 0.613  
ls_004 | author | broad    | ['—']   | 0.00 | 0.00 | 0.000 | 0.000  

── L1.1 output（依 LitSearch 慣例彙總）
{
  "specific_R@5": 0.75,
  "specific_R@20": 0.75,
  "broad_R@20": 0.5,
  "MRR": 0.411,
  "nDCG@10": 0.394
}



### 分層回報，並與論文 baseline 對照

LitSearch 論文把結果切成 inline / author × broad / specific 四格回報（另可再依 Qual = 1 / 2 分）。下面的 baseline 數字取自論文 Table 3 和 Table 7。商用引擎那組跟我們的 Step 2 一樣打線上搜尋，論文自己也註明這種比較不是 apples-to-apples。

In [6]:
rows = []
for t in ["inline", "author"]:
    for s in ["broad", "specific"]:
        qs = [q for q in litsearch if q["type"] == t and q["spec"] == s]
        metric = "R@20" if s == "broad" else "R@5"
        rows.append((t, s, len(qs), metric, f"{np.mean([q[metric] for q in qs]):.2f}"))
table(rows, ["type", "spec", "#Q", "主要指標", "值"])

BASELINES = {  # LitSearch Table 3 (Avg) 與 Table 7（80 題 specific 子集）
    "BM25": {"avg_broad_R@20": 39.9, "avg_specific_R@5": 50.0},
    "GritLM-7B": {"avg_broad_R@20": 70.8, "avg_specific_R@5": 74.8},
    "GPT-4o rerank (w/ GritLM)": {"avg_broad_R@20": 75.3, "avg_specific_R@5": 79.2},
    "Google Search (R@5 inline/author)": {"inline": 23.1, "author": 62.5},
    "Google Scholar (R@5 inline/author)": {"inline": 20.5, "author": 17.5},
    "Elicit (R@5 inline/author)": {"inline": 23.1, "author": 17.5},
}
ours = {"avg_broad_R@20": round(100 * l1["broad_R@20"], 1), "avg_specific_R@5": round(100 * l1["specific_R@5"], 1)}
show("baseline（%）", BASELINES)
show("ours（toy，%）", ours)

type   | spec     | #Q | 主要指標 | 值   
-------+----------+----+------+-----
inline | broad    | 1  | R@20 | 1.00
inline | specific | 1  | R@5  | 1.00
author | broad    | 1  | R@20 | 0.00
author | specific | 1  | R@5  | 0.50

── baseline（%）
{
  "BM25": {
    "avg_broad_R@20": 39.9,
    "avg_specific_R@5": 50.0
  },
  "GritLM-7B": {
    "avg_broad_R@20": 70.8,
    "avg_specific_R@5": 74.8
  },
  "GPT-4o rerank (w/ GritLM)": {
    "avg_broad_R@20": 75.3,
    "avg_specific_R@5": 79.2
  },
  "Google Search (R@5 inline/author)": {
    "inline": 23.1,
    "author": 62.5
  },
  "Google Scholar (R@5 inline/author)": {
    "inline": 20.5,
    "author": 17.5
  },
  "Elicit (R@5 inline/author)": {
    "inline": 23.1,
    "author": 17.5
  }
}

── ours（toy，%）
{
  "avg_broad_R@20": 50.0,
  "avg_specific_R@5": 75.0
}



## L1.2 Step 2 只能算 set recall

`step2_raw_papers.json` 依「query 編號 → 標題」排序，**不是相關度順序**，所以不能算 R@k。Step 2 只能問「gold 有沒有在集合裡」，而且必須同時回報集合大小 $|S_q|$，因為把集合做大一定能拉高 recall。

$$\text{SetRecall}(q)=\frac{|G_q\cap S_q|}{|G_q|}$$

In [7]:
step2_set = {  # 每題 Step 2 撈回來的論文集合（toy）
    "ls_001": set(litsearch[0]["ranked"]) | {f"y{i}" for i in range(80)},
    "ls_002": set(litsearch[1]["ranked"]) | {f"y{i}" for i in range(40)},
    "ls_003": set(litsearch[2]["ranked"]) | {f"y{i}" for i in range(100)},
    "ls_004": {f"y{i}" for i in range(120)},  # gold 根本沒被搜到
}
rows = []
for q in litsearch:
    S = step2_set[q["qid"]]
    rows.append((q["qid"], len(S), f"{len(set(q['gold']) & S) / len(q['gold']):.2f}"))
table(rows, ["qid", "|S_q|", "SetRecall"])
l1["step2_set_recall"] = round(np.mean([len(set(q["gold"]) & step2_set[q["qid"]]) / len(q["gold"]) for q in litsearch]), 3)
l1["step2_mean_set_size"] = float(np.mean([len(s) for s in step2_set.values()]))
show("L1.2 output", {k: l1[k] for k in ["step2_set_recall", "step2_mean_set_size"]})

qid    | |S_q| | SetRecall
-------+-------+----------
ls_001 | 110   | 1.00     
ls_002 | 70    | 1.00     
ls_003 | 130   | 1.00     
ls_004 | 120   | 0.00     

── L1.2 output
{
  "step2_set_recall": 0.75,
  "step2_mean_set_size": 107.5
}



## L1.3 Step 3 排序：composite、relevance-only，以及同分的期望值

Step 3 的 composite 由 LLM 給的整數分數加權而成：

$$\text{composite}=0.5\cdot\text{rel}+0.3\cdot\text{qual}+0.2\cdot\text{rec},\qquad \text{rel},\text{qual},\text{rec}\in\{1,\dots,5\}$$

composite 混了 quality 和 recency，不純是相關度，所以要另外算一個**只用 relevance 排序**的版本。

整數分數會產生大量同分。假設 gold 落在一個同分群組裡，這個群組佔據名次 $a..b$（大小 $t=b-a+1$），並且同分時隨機排序，則：

$$P(\text{rank}\le k)=\frac{\max\big(0,\ \min(k,b)-a+1\big)}{t},\qquad \mathbb E[\text{RR}]=\frac{1}{t}\sum_{p=a}^{b}\frac{1}{p}$$

這樣算出來的結果不受 tie-break 規則（例如依 paper id 排）影響。

In [8]:
# input：Step 3 對某一題候選論文的 LLM 整數分數（toy）
scores = [  # (paper_id, relevance, quality, recency_impact)
    ("p_a", 5, 5, 4), ("p_b", 5, 4, 4), ("p_c", 4, 5, 5),
    ("GOLD", 4, 4, 4), ("p_d", 4, 4, 4), ("p_e", 4, 4, 4), ("p_f", 4, 4, 4),
    ("p_g", 3, 5, 5), ("p_h", 3, 3, 3),
]
W = {"rel": Decimal("0.5"), "qual": Decimal("0.3"), "rec": Decimal("0.2")}


def composite(rel, qual, rec):
    return W["rel"] * rel + W["qual"] * qual + W["rec"] * rec


rows = sorted(((pid, float(composite(r, q, c)), r) for pid, r, q, c in scores), key=lambda x: (-x[1], x[0]))
table([(i, pid, f"{c:.1f}", rel) for i, (pid, c, rel) in enumerate(rows, 1)], ["名次", "paper", "composite", "relevance"])


def tie_group(sorted_rows, target, key_idx):
    """Return (a, b) = 1-based positions occupied by target's tie group."""
    val = next(r[key_idx] for r in sorted_rows if r[0] == target)
    pos = [i for i, r in enumerate(sorted_rows, 1) if r[key_idx] == val]
    return pos[0], pos[-1]


def expected_hit_and_rr(a, b, k):
    t = b - a + 1
    return max(0, min(k, b) - a + 1) / t, sum(1 / p for p in range(a, b + 1)) / t


a, b = tie_group(rows, "GOLD", 1)
p5, err = expected_hit_and_rr(a, b, 5)
naive_rank = [r[0] for r in rows].index("GOLD") + 1

rows_rel = sorted(rows, key=lambda x: (-x[2], x[0]))
a2, b2 = tie_group(rows_rel, "GOLD", 2)
p5_rel, err_rel = expected_hit_and_rr(a2, b2, 5)

show("L1.3 output", {
    "composite: GOLD 同分群組名次": [a, b],
    "依 paper_id tie-break 的名次（會被字母順序左右）": naive_rank,
    "composite: E[1(rank<=5)]": round(p5, 3), "composite: E[RR]": round(err, 3),
    "relevance-only: 同分群組名次": [a2, b2],
    "relevance-only: E[1(rank<=5)]": round(p5_rel, 3), "relevance-only: E[RR]": round(err_rel, 3),
})

名次 | paper | composite | relevance
---+-------+-----------+----------
1  | p_a   | 4.8       | 5        
2  | p_b   | 4.5       | 5        
3  | p_c   | 4.5       | 4        
4  | GOLD  | 4.0       | 4        
5  | p_d   | 4.0       | 4        
6  | p_e   | 4.0       | 4        
7  | p_f   | 4.0       | 4        
8  | p_g   | 4.0       | 3        
9  | p_h   | 3.0       | 3        

── L1.3 output
{
  "composite: GOLD 同分群組名次": [
    4,
    8
  ],
  "依 paper_id tie-break 的名次（會被字母順序左右）": 4,
  "composite: E[1(rank<=5)]": 0.4,
  "composite: E[RR]": 0.177,
  "relevance-only: 同分群組名次": [
    3,
    7
  ],
  "relevance-only: E[1(rank<=5)]": 0.6,
  "relevance-only: E[RR]": 0.219
}



## L1.4 Gold ID 對映

LitSearch 的 gold 以 S2ORC `corpusId` 標註；我們的紀錄有 `doi`、`semantic_scholar_id`、`arxiv_id` 和 `title`。對映時依序嘗試 DOI → arXiv → S2 paperId，都對不到才用標題比對。標題比對沿用引擎 `utils.word_overlap` 的定義（只算長度 > 2 的詞）：

$$\text{overlap}(a,b)=\frac{|T(a)\cap T(b)|}{\min(|T(a)|,|T(b)|)},\qquad \text{match} \iff \text{overlap}\ge 0.85$$

`corpusId → paperId / DOI / arXiv` 的轉換可以用 S2 API（`CorpusId:xxx`）先建好一張表並 cache。

In [9]:
def title_tokens(t):
    t = re.sub(r"[^\w\s]", " ", t.lower())
    return {w for w in t.split() if len(w) > 2}


def word_overlap(a, b):
    ta, tb = title_tokens(a), title_tokens(b)
    return len(ta & tb) / min(len(ta), len(tb)) if ta and tb else 0.0


gold_records = [  # 先用 S2 API 把 corpusId 轉好（toy）
    {"corpusId": 230433941, "doi": "10.18653/v1/2021.acl-long.353", "arxiv": "2101.00190", "title": "Prefix-Tuning: Optimizing Continuous Prompts for Generation"},
    {"corpusId": 263830786, "doi": None, "arxiv": None, "title": "Sheared LLaMA: Accelerating Language Model Pre-training via Structured Pruning"},
]
our_records = [  # step2_raw_papers.json 的一部分（toy）
    {"id": "paper_014", "doi": None, "arxiv_id": "2101.00190", "semantic_scholar_id": "53d8b35...", "title": "Prefix-Tuning: Optimizing Continuous Prompts for Generation"},
    {"id": "paper_031", "doi": None, "arxiv_id": None, "semantic_scholar_id": "f2a1...", "title": "Sheared LLaMA: accelerating language model pre-training via structured pruning."},
    {"id": "paper_032", "doi": None, "arxiv_id": None, "semantic_scholar_id": "aa11...", "title": "LLM-Pruner: On the Structural Pruning of Large Language Models"},
]


def map_gold(g, ours):
    for r in ours:
        if g["doi"] and r["doi"] and g["doi"].lower() == r["doi"].lower():
            return r["id"], "doi"
        if g["arxiv"] and r["arxiv_id"] == g["arxiv"]:
            return r["id"], "arxiv"
    best = max(ours, key=lambda r: word_overlap(g["title"], r["title"]))
    ov = word_overlap(g["title"], best["title"])
    return (best["id"], f"title overlap={ov:.2f}") if ov >= 0.85 else (None, f"no match (best {ov:.2f})")


table([(g["corpusId"], *map_gold(g, our_records)) for g in gold_records], ["gold corpusId", "our id", "matched by"])

gold corpusId | our id    | matched by        
--------------+-----------+-------------------
230433941     | paper_014 | arxiv             
263830786     | paper_031 | title overlap=1.00



## L1.5 損失漏斗：gold 在哪一關消失

對每篇 gold paper，記錄它是否通過各關，**第一個沒通過的關卡**就是它消失的原因。依 pipeline 順序：

`provider 搜到` → `年份過濾` → `dedup 沒被誤合併` → `snowball overlap 過濾`（只適用於 snowball 來源）→ `Step 3 沒被排除` → `名次 ≤ k`

In [10]:
STAGES = ["retrieved_by_provider", "passed_year_filter", "survived_dedup", "passed_snowball_overlap", "kept_in_step3", "rank<=20"]
gold_trace = [  # toy：每篇 gold 在各關的結果（True = 通過）
    {"gold": "PrefixTuning", "retrieved_by_provider": True, "passed_year_filter": True, "survived_dedup": True, "passed_snowball_overlap": True, "kept_in_step3": True, "rank<=20": True},
    {"gold": "BERT", "retrieved_by_provider": True, "passed_year_filter": False},  # 2019 < timeframe 起始年
    {"gold": "ShearedLLaMA", "retrieved_by_provider": True, "passed_year_filter": True, "survived_dedup": True, "passed_snowball_overlap": True, "kept_in_step3": True, "rank<=20": True},
    {"gold": "CoFiPruning", "retrieved_by_provider": True, "passed_year_filter": True, "survived_dedup": True, "passed_snowball_overlap": True, "kept_in_step3": True, "rank<=20": False},
    {"gold": "SQuADv2", "retrieved_by_provider": False},
    {"gold": "LoRA", "retrieved_by_provider": True, "passed_year_filter": True, "survived_dedup": True, "passed_snowball_overlap": True, "kept_in_step3": False, "exclusion_reason": "off-PICO: not about pruning"},
]


def lost_at(tr):
    for s in STAGES:
        if not tr.get(s, False):
            return s
    return "found"


funnel = Counter(lost_at(t) for t in gold_trace)
table([(t["gold"], lost_at(t), t.get("exclusion_reason", "")) for t in gold_trace], ["gold", "lost_at", "Step 3 exclusion_reason"])
show("L1.5 output（漏斗計數）", {s: funnel.get(s, 0) for s in STAGES + ["found"]})
l1["loss_funnel"] = {s: funnel.get(s, 0) for s in STAGES + ["found"]}

gold         | lost_at               | Step 3 exclusion_reason    
-------------+-----------------------+----------------------------
PrefixTuning | found                 |                            
BERT         | passed_year_filter    |                            
ShearedLLaMA | found                 |                            
CoFiPruning  | rank<=20              |                            
SQuADv2      | retrieved_by_provider |                            
LoRA         | kept_in_step3         | off-PICO: not about pruning

── L1.5 output（漏斗計數）
{
  "retrieved_by_provider": 1,
  "passed_year_filter": 1,
  "survived_dedup": 0,
  "passed_snowball_overlap": 0,
  "kept_in_step3": 1,
  "rank<=20": 1,
  "found": 2
}



## L1.6 Mode A vs Mode B

- **Mode A**：LitSearch 問題原文直接當 Q1，跳過 Step 1 的 LLM，只量檢索本身。
- **Mode B**：問題當 topic 餵給 `scholar-research new`，由 Step 1 自己產 PICO 和 query。

$$\Delta_{\text{Step 1}}=\overline{\text{R@20}}_{A}-\overline{\text{R@20}}_{B}$$

$\Delta>0$ 代表 Step 1 的 query 改寫讓檢索變差，這就是 query 生成造成的損失。

In [11]:
modeAB = {  # qid: (R@20 Mode A, R@20 Mode B)  — toy
    "ls_001": (1.0, 1.0), "ls_002": (1.0, 0.0), "ls_003": (0.5, 0.5), "ls_004": (0.0, 0.0),
    "ls_005": (1.0, 1.0), "ls_006": (1.0, 0.0), "ls_007": (0.0, 1.0), "ls_008": (1.0, 1.0),
}
A = np.array([v[0] for v in modeAB.values()])
B = np.array([v[1] for v in modeAB.values()])
l1["modeA_R@20"], l1["modeB_R@20"] = round(A.mean(), 3), round(B.mean(), 3)
l1["delta_step1"] = round(A.mean() - B.mean(), 3)
show("L1.6 output", {"Mode A": l1["modeA_R@20"], "Mode B": l1["modeB_R@20"], "Δ_Step1 = A − B": l1["delta_step1"],
                     "A 對 B 錯的題": [q for q, (a, b) in modeAB.items() if a > b],
                     "B 對 A 錯的題": [q for q, (a, b) in modeAB.items() if b > a]})

── L1.6 output
{
  "Mode A": 0.688,
  "Mode B": 0.562,
  "Δ_Step1 = A − B": 0.125,
  "A 對 B 錯的題": [
    "ls_002",
    "ls_006"
  ],
  "B 對 A 錯的題": [
    "ls_007"
  ]
}



## L1.7 主題層級 recall（借 IdeaBench 的 target paper → references 結構）

LitSearch 驗的是「找到特定一篇」。要驗「綜述該找的有沒有找到」，可以把 target paper 的主題當 topic 輸入，用它（過濾後）的 references 當 gold set $R$：

$$\text{TopicRecall}_{\text{Step2}}=\frac{|R\cap S|}{|R|},\qquad \text{TopicRecall}_{\text{Step3}}=\frac{|R\cap \text{Shortlist}|}{|R|},\qquad \text{Retention}=\frac{|R\cap \text{Shortlist}|}{|R\cap S|}$$

Retention 單獨看 Step 3 的篩選：Step 2 已經找到的 reference 中，有多少被 Step 3 留下。

In [12]:
refs_gold = {f"ref_{i:02d}" for i in range(1, 13)}                     # target paper 的 12 篇 background references
S = {f"ref_{i:02d}" for i in [1, 2, 3, 4, 5, 6, 7, 9, 11]} | {f"z{i}" for i in range(150)}
shortlist = {f"ref_{i:02d}" for i in [1, 2, 3, 5, 9]} | {f"z{i}" for i in range(25)}
tr = {
    "TopicRecall_step2": round(len(refs_gold & S) / len(refs_gold), 3),
    "TopicRecall_step3": round(len(refs_gold & shortlist) / len(refs_gold), 3),
    "Step3_retention": round(len(refs_gold & shortlist) / len(refs_gold & S), 3),
    "|S|": len(S), "|shortlist|": len(shortlist),
}
show("L1.7 output", tr)
l1["topic_recall"] = tr
REPORT["L1_retrieval"] = l1

── L1.7 output
{
  "TopicRecall_step2": 0.75,
  "TopicRecall_step3": 0.417,
  "Step3_retention": 0.556,
  "|S|": 159,
  "|shortlist|": 30
}



---
# L2 Step 6 確定性檢查（不需要 GPU 或 LLM）

## Toy session

9 篇論文，2 個主題。`ACCESS` 是 Step 5 的 `access_level`；`FULLTEXT` 是 `step5_full_text/{key}.md` 的正文（極度簡化）；`SOTA` 的結構與 `step6_sota.json` 相同（只列會用到的欄位）。

In [13]:
ACCESS = {
    "YamadaEtAl2025": "full-text-pdf", "RavideshikEtAl2024": "full-text-pdf", "ChenEtAl2024": "full-text-html",
    "GaddipatiLee2023": "full-text-pdf", "Park2022": "abstract-only", "LiuEtAl2021a": "full-text-pdf",
    "LiuEtAl2021b": "partial-text-html", "Okafor2023": "full-text-pdf", "SantosEtAl2022": "full-text-pdf",
}
KEYS = set(ACCESS)  # 合法鍵集合 K（= step4_citation_keys.json 的所有 key）

FULLTEXT = {
    "YamadaEtAl2025": "Forty-eight patients (N = 48) with MCI completed 20 sessions of theta/beta neurofeedback. MoCA improved by 2.4 points relative to baseline (p = 0.003).",
    "RavideshikEtAl2024": "In a sham-controlled trial of 60 patients, the MoCA difference between active and sham neurofeedback was 0.3 points (p = 0.61), which was not significant.",
    "ChenEtAl2024": "Alpha-band neurofeedback did not improve memory scores (d = 0.12). Baseline alpha power did not predict response.",
    "GaddipatiLee2023": "A gradient-boosting classifier discriminated responders from non-responders with 91.5% accuracy.",
    "Park2022": "We review neurofeedback protocols used in older adults.",
    "LiuEtAl2021a": "Resting-state EEG was recorded before training in 32 participants. Cognitive gain increased with session count (r = 0.29).",
    "LiuEtAl2021b": "Baseline alpha power showed a weak association with response in a subset of participants.",
    "Okafor2023": "We compared home-based and clinic-based neurofeedback adherence.",
    "SantosEtAl2022": "EEG microstate changes after neurofeedback in 25 older adults.",
}

SOTA = {"themes": [
    {"number": 1, "title_en": "Cognitive efficacy of EEG neurofeedback",
     "paper_keys": ["YamadaEtAl2025", "RavideshikEtAl2024", "ChenEtAl2024", "Okafor2023", "Park2022"],
     "consensus_en": "Theta/beta neurofeedback improved global cognition in MCI (Yamada et al., 2025; Liu et al., 2021a). Yamada et al. (2025) reported a 2.4-point MoCA gain after 20 sessions in 48 patients.",
     "consensus_zh": "Theta/beta 神經回饋改善了 MCI 的整體認知（Yamada et al., 2025；Liu et al., 2021a）。Yamada et al. (2025) 報告 48 名患者在 20 次訓練後 MoCA 提升 2.4 分。",
     "debates_en": "However, a sham-controlled trial found no significant MoCA difference (p = 0.61) (Ravideshik et al., 2024). Alpha-band protocols also failed to improve memory (Chen et al., 2024).",
     "debates_zh": "然而，一項假回饋對照試驗發現 MoCA 無顯著差異（p = 0.61）（Ravideshik et al., 2024）。",
     "methods_en": "Most studies used sham-controlled designs with 20–40 sessions. We now turn to the reported effect sizes.",
     "methods_zh": "多數研究採假回饋對照設計，訓練 20–40 次。",
     "key_results_en": "| Study | N | Method | Primary Outcome | Result |\n|---|---|---|---|---|\n| YamadaEtAl2025 | 48 | theta/beta NF | MoCA | +2.4 (p = 0.003) |\n| RavideshikEtAl2024 | 60 | sham-controlled NF | MoCA | +0.3 (p = 0.61) |",
     "key_results_zh": "（同英文表格）"},
    {"number": 2, "title_en": "Predictors of treatment response",
     "paper_keys": ["GaddipatiLee2023", "LiuEtAl2021a", "LiuEtAl2021b", "SantosEtAl2022"],
     "consensus_en": "Session count correlated with cognitive gain (r = +0.29) (Liu et al., 2021a). Responders could be discriminated with 80% accuracy (Gaddipati & Lee, 2023; Park, 2022).",
     "consensus_zh": "訓練次數與認知進步相關（r = +0.29）（Liu et al., 2021a）。反應者可被以 80% 準確率區分（Gaddipati & Lee, 2023；Park, 2022）。",
     "debates_en": "Whether baseline alpha power predicts response remains unresolved (Chen et al., 2024; Liu et al., 2021b).",
     "debates_zh": "基線 alpha 功率能否預測反應仍無定論（Chen et al., 2024；Liu et al., 2021b）。",
     "methods_en": "Predictor studies relied on resting-state EEG recorded before training (Liu et al., 2021a).",
     "methods_zh": "",
     "key_results_en": "Taken together, predictor evidence remains preliminary.",
     "key_results_zh": "整體而言，預測因子的證據仍屬初步。"},
]}
# paper_map：經過引擎 _repair() 之後，每篇都一定有主題
SOTA["paper_map"] = [{"citation_key": k, "theme_number": t["number"]} for t in SOTA["themes"] for k in t["paper_keys"]]
N_REPAIRED = 2  # 假設 LLM 原始輸出漏了 Okafor2023、SantosEtAl2022，由 _repair() 補進去
DIMS = ["consensus", "debates", "methods", "key_results"]
print(f"{len(KEYS)} keys, {len(SOTA['themes'])} themes, paper_map={len(SOTA['paper_map'])} rows")

9 keys, 2 themes, paper_map=9 rows


## L2.1 Citation key 正規化

Step 4 的 key 規則（`s4_export.citation_keys`）：1 位作者為 `Park2022`；2 位作者為 `GaddipatiLee2023`；3 位以上為 `YamadaEtAl2025`；撞名時加 `a` / `b` 後綴（`LiuEtAl2021a`）。

Step 6 的 prompt 要求行內寫 `(AuthorYear)`，但模型實際可能寫成下面幾種形式，全部要對回同一個 key：

| 寫法 | 正規化結果 |
|---|---|
| `(Yamada et al., 2025)` | `YamadaEtAl2025` |
| `Yamada et al. (2025)`（敘述式） | `YamadaEtAl2025` |
| `(Gaddipati & Lee, 2023)` / `(Gaddipati and Lee 2023)` | `GaddipatiLee2023` |
| `(Park, 2022)` | `Park2022` |
| `(YamadaEtAl2025)`（直接寫 key） | `YamadaEtAl2025` |
| `(Yamada et al., 2025; Liu et al., 2021a)` | 兩個 key |
| 中文全形 `（…；…）` | 先轉成半形再處理 |

In [14]:
NAME = r"[A-Z][A-Za-z\-']+"
NARRATIVE_RE = re.compile(rf"({NAME})(?:\s+et al\.|\s+(?:&|and)\s+({NAME}))?\s+\((\d{{4}}[a-z]?)\)")
PAREN_RE = re.compile(r"\(([^()]*)\)")
PART_RE = re.compile(rf"^\s*({NAME})(?:\s+et al\.?|\s*(?:&|and)\s*({NAME}))?,?\s+(\d{{4}}[a-z]?)\s*$")
RAWKEY_RE = re.compile(r"^\s*([A-Z][A-Za-z\-]*\d{4}[a-z]?)\s*$")


def to_key(first, second, year, et_al):
    if et_al:
        return f"{first}EtAl{year}"
    return f"{first}{second or ''}{year}"


def normalize_fullwidth(text):
    return text.replace("（", "(").replace("）", ")").replace("；", ";").replace("，", ",")


def extract_citations(text):
    """Return (keys_in_order, text_with_citations_removed)."""
    text = normalize_fullwidth(text)
    keys = []

    def narr(m):
        et_al = "et al" in m.group(0)
        keys.append(to_key(m.group(1), m.group(2), m.group(3), et_al))
        return m.group(1)  # 保留作者名讓句子仍通順，年份拿掉

    text = NARRATIVE_RE.sub(narr, text)

    def paren(m):
        found = []
        for part in m.group(1).split(";"):
            if (r := RAWKEY_RE.match(part)):
                found.append(r.group(1))
            elif (p := PART_RE.match(part)):
                found.append(to_key(p.group(1), p.group(2), p.group(3), "et al" in part))
        if not found:
            return m.group(0)  # 不是引用，例如 (p = 0.61)
        keys.extend(found)
        return ""

    text = PAREN_RE.sub(paren, text)
    return keys, re.sub(r"\s+([.;,])", r"\1", re.sub(r"\s{2,}", " ", text)).strip()


examples = ["(Yamada et al., 2025)", "Yamada et al. (2025) reported", "(Gaddipati & Lee, 2023)", "(Gaddipati and Lee 2023)",
            "(Park, 2022)", "(YamadaEtAl2025)", "(Yamada et al., 2025; Liu et al., 2021a)", "（Yamada et al., 2025；Liu et al., 2021a）",
            "no significant difference (p = 0.61)"]
table([(e, extract_citations(e)[0]) for e in examples], ["寫法", "keys"])

寫法                                       | keys                              
-----------------------------------------+-----------------------------------
(Yamada et al., 2025)                    | ['YamadaEtAl2025']                
Yamada et al. (2025) reported            | ['YamadaEtAl2025']                
(Gaddipati & Lee, 2023)                  | ['GaddipatiLee2023']              
(Gaddipati and Lee 2023)                 | ['GaddipatiLee2023']              
(Park, 2022)                             | ['Park2022']                      
(YamadaEtAl2025)                         | ['YamadaEtAl2025']                
(Yamada et al., 2025; Liu et al., 2021a) | ['YamadaEtAl2025', 'LiuEtAl2021a']
（Yamada et al., 2025；Liu et al., 2021a）  | ['YamadaEtAl2025', 'LiuEtAl2021a']
no significant difference (p = 0.61)     | []                                



## L2.2 Citation integrity（硬閘）

令 $C$ 為 SOTA 正文（四個維度，含 Key Results 表格）抽出的所有 key，$K$ 為合法鍵集合：

$$\text{integrity\_fail}=\mathbb 1\left[C\not\subseteq K\right]$$

一旦 `integrity_fail = true`，Cite-p、Cite-r、Cite-F1 全部記 0，不必再跑 LLM judge。

同一個檢查也要套在 `step7_gaps.json` 的 evidence 上（引擎的 `or g.supporting_evidence` 會在全部都是幽靈鍵時原封保留）。另外做一個 **EN/ZH 一致性**檢查：同一段落的中英文版本，key 集合應該相同。

In [15]:
def table_study_keys(md_text):
    """Key Results 表格：Study 欄就是引用。"""
    keys = []
    for line in md_text.splitlines():
        cells = [c.strip() for c in line.strip().strip("|").split("|")]
        if len(cells) >= 5 and cells[0] not in ("Study", "") and not set(cells[0]) <= set("-: "):
            k, _ = extract_citations(f"({cells[0]})")
            keys.extend(k or [cells[0]])
    return keys


def section_keys(theme, dim, lang="en"):
    text = theme[f"{dim}_{lang}"]
    if dim == "key_results" and text.lstrip().startswith("|"):
        return table_study_keys(text)
    return extract_citations(text)[0]


def integrity(sota, K):
    cited = {(t["number"], d, k) for t in sota["themes"] for d in DIMS for k in section_keys(t, d)}
    phantoms = sorted({(n, d, k) for n, d, k in cited if k not in K})
    return {"n_cited_keys": len({k for *_, k in cited}), "integrity_fail": bool(phantoms),
            "phantoms": [f"theme{n}.{d}: {k}" for n, d, k in phantoms]}


show("正常版本", integrity(SOTA, KEYS))

SOTA_BAD = json.loads(json.dumps(SOTA))
SOTA_BAD["themes"][1]["consensus_en"] += " Accuracy later rose to 95% (Fake et al., 2024)."
show("插入幽靈鍵的版本", integrity(SOTA_BAD, KEYS))

STEP7_GAPS = [  # step7_gaps.json 的 evidence（toy）
    {"id": "GAP_001", "supporting": ["YamadaEtAl2025", "RavideshikEtAl2024"], "counter": ["ChenEtAl2024"]},
    {"id": "GAP_002", "supporting": ["Ghost2020", "NobodyEtAl2019"], "counter": []},  # 全部是幽靈鍵，引擎目前會保留
]
show("Step 7 evidence integrity", [{"gap": g["id"], "phantoms": [k for k in g["supporting"] + g["counter"] if k not in KEYS]} for g in STEP7_GAPS])

rows = []
for t in SOTA["themes"]:
    for d in DIMS:
        en, zh = set(section_keys(t, d, "en")), set(section_keys(t, d, "zh"))
        if d == "key_results" and t[f"{d}_zh"].startswith("（同英文"):
            continue
        if en != zh:
            rows.append((f"theme{t['number']}.{d}", sorted(en - zh), sorted(zh - en)))
print("EN/ZH 不一致的段落：")
table(rows, ["段落", "只在 EN", "只在 ZH"])

── 正常版本
{
  "n_cited_keys": 7,
  "integrity_fail": false,
  "phantoms": []
}

── 插入幽靈鍵的版本
{
  "n_cited_keys": 8,
  "integrity_fail": true,
  "phantoms": [
    "theme2.consensus: FakeEtAl2024"
  ]
}

── Step 7 evidence integrity
[
  {
    "gap": "GAP_001",
    "phantoms": []
  },
  {
    "gap": "GAP_002",
    "phantoms": [
      "Ghost2020",
      "NobodyEtAl2019"
    ]
  }
]

EN/ZH 不一致的段落：
段落             | 只在 EN            | 只在 ZH
---------------+------------------+------
theme1.debates | ['ChenEtAl2024'] | []   
theme2.methods | ['LiuEtAl2021a'] | []   



## L2.3 結構檢查（organization 的確定性部分）

對每個主題 $t$，檢查：

1. 四個維度（Consensus / Debates / Dominant Methods / Key Results）的中英文都不能是空的。
2. $|\text{keys}(\text{Consensus})|\ge 2$（共識至少由 2 篇支持）。
3. $|\text{keys}(\text{Debates})|\ge 2$（爭議的雙方都有引用）。
4. Key Results 若是表格，表頭必須為 `| Study | N | Method | Primary Outcome | Result |`。
5. 每個主題至少 2 篇論文。
6. 主題數要落在 prompt 規定的範圍內：論文 < 10 篇時 2–3 個；10–30 篇時 4–6 個；> 30 篇時最多 8 個。

In [16]:
HEADER = "| Study | N | Method | Primary Outcome | Result |"


def theme_range(n_papers):
    return (2, 3) if n_papers < 10 else ((4, 6) if n_papers <= 30 else (4, 8))


def structure_checks(sota, n_papers):
    issues = []
    lo, hi = theme_range(n_papers)
    if not lo <= len(sota["themes"]) <= hi:
        issues.append(f"theme count {len(sota['themes'])} not in [{lo},{hi}]")
    for t in sota["themes"]:
        n = t["number"]
        for d in DIMS:
            for lang in ("en", "zh"):
                if not t[f"{d}_{lang}"].strip():
                    issues.append(f"theme{n}.{d}_{lang} is empty")
        if len(set(section_keys(t, "consensus"))) < 2:
            issues.append(f"theme{n}.consensus cites <2 papers")
        if len(set(section_keys(t, "debates"))) < 2:
            issues.append(f"theme{n}.debates cites <2 papers")
        kr = t["key_results_en"].strip()
        if kr.startswith("|") and kr.splitlines()[0].strip() != HEADER:
            issues.append(f"theme{n}.key_results table header mismatch")
        if len(t["paper_keys"]) < 2:
            issues.append(f"theme{n} has <2 papers")
    return {"structure_pass": not issues, "issues": issues}


l2_structure = structure_checks(SOTA, len(KEYS))
show("L2.3 output", l2_structure)

── L2.3 output
{
  "structure_pass": false,
  "issues": [
    "theme2.methods_zh is empty"
  ]
}



## L2.4 ≥ 80% 全文覆蓋

依 Joanna 9/20 的定義：

$$F_A=\{p:\ \text{access\_level}(p)\in A\},\qquad \text{Coverage}_A=\frac{\big|\{p\in F_A:\ p \text{ 在至少一個主題的正文或 Key Results 表格中被引用}\}\big|}{|F_A|}$$

- 跨主題的論文只算一次；表格引用和正文引用同等。
- $A=\{\texttt{full-text-pdf}\}$ 是 Joanna 的定義；$A=\{\texttt{full-text-pdf},\texttt{full-text-html}\}$ 是 plan 建議的版本，**週四要定**。
- 過關條件：$\text{Coverage}_A\ge 0.80$。

⚠ **不能用 `paper_map` 判定**：引擎的 `_repair()` 會把每篇論文都塞進某個主題，用它算永遠是 100%。下面三種算法並列，就能看出差異。

In [17]:
cited_anywhere = {k for t in SOTA["themes"] for d in DIMS for k in section_keys(t, d)}
in_paper_map = {m["citation_key"] for m in SOTA["paper_map"]}


def coverage(levels, assigned):
    F = {k for k, lv in ACCESS.items() if lv in levels}
    hit = F & assigned
    return {"denominator": sorted(F), "uncovered": sorted(F - hit), "coverage": round(len(hit) / len(F), 3), "pass(>=0.80)": len(hit) / len(F) >= 0.80}


l2_cov = {
    "paper_map 版（恆為 100%，不可用）": coverage({"full-text-pdf"}, in_paper_map),
    "cited 版, A={pdf}（Joanna 定義）": coverage({"full-text-pdf"}, cited_anywhere),
    "cited 版, A={pdf, html}（plan 建議）": coverage({"full-text-pdf", "full-text-html"}, cited_anywhere),
    "n_repaired（_repair 補進的篇數）": N_REPAIRED,
}
show("L2.4 output", l2_cov)

── L2.4 output
{
  "paper_map 版（恆為 100%，不可用）": {
    "denominator": [
      "GaddipatiLee2023",
      "LiuEtAl2021a",
      "Okafor2023",
      "RavideshikEtAl2024",
      "SantosEtAl2022",
      "YamadaEtAl2025"
    ],
    "uncovered": [],
    "coverage": 1.0,
    "pass(>=0.80)": true
  },
  "cited 版, A={pdf}（Joanna 定義）": {
    "denominator": [
      "GaddipatiLee2023",
      "LiuEtAl2021a",
      "Okafor2023",
      "RavideshikEtAl2024",
      "SantosEtAl2022",
      "YamadaEtAl2025"
    ],
    "uncovered": [
      "Okafor2023",
      "SantosEtAl2022"
    ],
    "coverage": 0.667,
    "pass(>=0.80)": false
  },
  "cited 版, A={pdf, html}（plan 建議）": {
    "denominator": [
      "ChenEtAl2024",
      "GaddipatiLee2023",
      "LiuEtAl2021a",
      "Okafor2023",
      "RavideshikEtAl2024",
      "SantosEtAl2022",
      "YamadaEtAl2025"
    ],
    "uncovered": [
      "Okafor2023",
      "SantosEtAl2022"
    ],
    "coverage": 0.714,
    "pass(>=0.80)": false
  },
  "n_repaired（_repair 補進

---
# L3 Step 6 引用品質：Cite-p / Cite-r / F1（ScholarQABench + Cite flow）

## L3.1 切成評分單位

一個單位 = `{unit_id, claim_text, keys[], source}`。

- **正文**：依句號或分號切句，但括號內不切（`(Yamada et al., 2025; Liu et al., 2021a)` 裡的分號不算），`et al.` 的句點也不算。引用掛在所在的那一句。
- **Key Results 表**：一列一筆，Result 等欄位是主張，Study 欄是引用。
- **不進 Cite**：Papers 名單、Paper–Theme Mapping、主題標題、方法學圖例、限制章節。
- 只評英文，中文由 L2.2 的一致性檢查負責。

In [18]:
def split_sentences(text):
    """Split on . ; 。 ； at paren depth 0 when followed by whitespace/end; never after 'et al'."""
    out, buf, depth = [], [], 0
    for i, ch in enumerate(text):
        buf.append(ch)
        depth += ch in "(（"
        depth -= ch in ")）"
        nxt = text[i + 1] if i + 1 < len(text) else " "
        if ch in ".;。；" and depth == 0 and nxt.isspace() and not "".join(buf).endswith("et al."):
            out.append("".join(buf).strip())
            buf = []
    if "".join(buf).strip():
        out.append("".join(buf).strip())
    return out


def make_units(sota):
    units = []
    for t in sota["themes"]:
        for d in DIMS:
            text = t[f"{d}_en"]
            if d == "key_results" and text.lstrip().startswith("|"):
                for line in text.splitlines()[2:]:
                    c = [x.strip() for x in line.strip().strip("|").split("|")]
                    claim = f"N = {c[1]}; {c[2]}; {c[3]}; {c[4]}"
                    units.append({"source": f"theme{t['number']}.{d}.table", "raw": line, "claim": claim, "keys": table_study_keys(line + "\n")})
                continue
            for s in split_sentences(text):
                keys, stripped = extract_citations(s)
                units.append({"source": f"theme{t['number']}.{d}", "raw": s, "claim": stripped, "keys": keys})
    for i, u in enumerate(units, 1):
        u["unit_id"] = f"u{i:02d}"
    return units


UNITS = make_units(SOTA)
table([(u["unit_id"], u["source"], u["claim"][:70] + ("…" if len(u["claim"]) > 70 else ""), u["keys"]) for u in UNITS], ["unit", "source", "claim (citations stripped)", "keys"])

unit | source                   | claim (citations stripped)                                              | keys                              
-----+--------------------------+-------------------------------------------------------------------------+-----------------------------------
u01  | theme1.consensus         | Theta/beta neurofeedback improved global cognition in MCI.              | ['YamadaEtAl2025', 'LiuEtAl2021a']
u02  | theme1.consensus         | Yamada reported a 2.4-point MoCA gain after 20 sessions in 48 patients… | ['YamadaEtAl2025']                
u03  | theme1.debates           | However, a sham-controlled trial found no significant MoCA difference … | ['RavideshikEtAl2024']            
u04  | theme1.debates           | Alpha-band protocols also failed to improve memory.                     | ['ChenEtAl2024']                  
u05  | theme1.methods           | Most studies used sham-controlled designs with 20–40 sessions.          | []                                

## L3.2 Citable 判定（規則版）

$$\text{citable}(u)=\big(\text{has\_number}(u)\ \lor\ \text{has\_reporting\_verb}(u)\ \lor\ \text{has\_causal}(u)\big)\ \land\ \lnot\,\text{non\_citable\_pattern}(u)$$

- **要進 Cite**：數字、百分比、p 值、相關係數；「X 做了／發現／報告」；對立結果；因果陳述。
- **不進 Cite**：純過渡句（例如 "We now turn to…"）、總結句（"Taken together…"）、本綜述的限制。

citable = 0 的單位不進任何分母：既不算漏引，也不算精準。實務上可以換成小分類器或 LLM 判斷，但規則版可重現，適合先用。

In [19]:
NUM_RE = re.compile(r"(?<![A-Za-z])[+\-−]?\d+(?:\.\d+)?%?")
REPORT_VERBS = re.compile(r"\b(found|finds?|report(?:ed|s)?|show(?:ed|s|n)?|demonstrat\w*|improv\w*|reduc\w*|increas\w*|decreas\w*|correlat\w*|predict\w*|fail\w*|did not|no significant|associat\w*|discriminat\w*|outperform\w*|used|relied)\b", re.I)
CAUSAL = re.compile(r"\b(because|therefore|led to|caus\w*|due to)\b", re.I)
NON_CITABLE = re.compile(r"^(we now turn|in this section|the following|taken together|next,)|\blimitations? of this review\b", re.I)


def is_citable(claim):
    if NON_CITABLE.search(claim.strip()):
        return False, "non-citable pattern"
    if NUM_RE.search(claim):
        return True, "number"
    if REPORT_VERBS.search(claim):
        return True, "reporting verb"
    if CAUSAL.search(claim):
        return True, "causal"
    return False, "no factual signal"


for u in UNITS:
    u["citable"], u["citable_reason"] = is_citable(u["claim"])
table([(u["unit_id"], int(u["citable"]), u["citable_reason"]) for u in UNITS], ["unit", "citable", "reason"])

unit | citable | reason             
-----+---------+--------------------
u01  | 1       | reporting verb     
u02  | 1       | number             
u03  | 1       | number             
u04  | 1       | reporting verb     
u05  | 1       | number             
u06  | 0       | non-citable pattern
u07  | 1       | number             
u08  | 1       | number             
u09  | 1       | number             
u10  | 1       | number             
u11  | 1       | reporting verb     
u12  | 1       | reporting verb     
u13  | 0       | non-citable pattern



## L3.3 數字 attestation（沿用 90% gate）

對每一個含數字的 citable 單位 $u$，抽出主張中的數字集合 $N(u)$（先把引用括號移除，年份才不會被當成數字）。只在**該單位所引 key 的全文**中搜尋，禁止全庫掃描；abstract-only 的 key 不能當證據。

**四捨五入等價**：主張中的數字 $x$ 寫到小數第 $d$ 位，全文中的數字為 $y$，則

$$x \equiv y \iff \text{round}(y, d) = x$$

正負號 `+` 視為正數，`%` 忽略。

$$\text{attested}(u)=\bigwedge_{x\in N(u)}\ \exists\, k\in \text{keys}(u),\ \text{level}(k)\neq\text{abstract-only},\ \exists\, y\in N(\text{fulltext}_k):\ x\equiv y$$

$$\text{NumericAttestation}=\frac{|\{u:\ \text{citable},\ N(u)\neq\varnothing,\ \text{attested}(u)\}|}{|\{u:\ \text{citable},\ N(u)\neq\varnothing\}|},\qquad \text{過關}\iff \ge 0.90$$

沒有引用的單位找不到證據，一律記為未通過。

In [19]:
def numbers(text):
    vals = []
    for tok in NUM_RE.findall(text):
        t = tok.replace("−", "-").replace("+", "").rstrip("%")
        d = len(t.split(".")[1]) if "." in t else 0
        vals.append((float(t), d, tok))
    return vals


def equiv(x, d, y):
    return abs(round(y, d) - x) < 1e-9


def attest(unit):
    claim_nums = numbers(unit["claim"])
    evidence = [k for k in unit["keys"] if k in FULLTEXT and ACCESS.get(k) != "abstract-only"]
    src_nums = [y for k in evidence for (y, _, _) in numbers(FULLTEXT[k])]
    missing = [tok for (x, d, tok) in claim_nums if not any(equiv(x, d, y) for y in src_nums)]
    return {"numbers": [tok for *_, tok in claim_nums], "evidence_keys": evidence, "missing": missing, "attested": not missing}


num_units = [u for u in UNITS if u["citable"] and numbers(u["claim"])]
for u in num_units:
    u["attest"] = attest(u)
table([(u["unit_id"], u["attest"]["numbers"], u["attest"]["evidence_keys"], u["attest"]["missing"], u["attest"]["attested"]) for u in num_units],
      ["unit", "claim numbers", "searched keys", "not found", "attested"])
numeric_attestation = sum(u["attest"]["attested"] for u in num_units) / len(num_units)
show("L3.3 output", {"numeric_attestation": round(numeric_attestation, 3), "pass(>=0.90)": numeric_attestation >= 0.90})

unit | claim numbers           | searched keys          | not found    | attested
-----+-------------------------+------------------------+--------------+---------
u02  | ['2.4', '20', '48']     | ['YamadaEtAl2025']     | []           | True    
u03  | ['0.61']                | ['RavideshikEtAl2024'] | []           | True    
u05  | ['20', '40']            | []                     | ['20', '40'] | False   
u07  | ['48', '+2.4', '0.003'] | ['YamadaEtAl2025']     | []           | True    
u08  | ['60', '+0.3', '0.61']  | ['RavideshikEtAl2024'] | []           | True    
u09  | ['+0.29']               | ['LiuEtAl2021a']       | []           | True    
u10  | ['80%']                 | ['GaddipatiLee2023']   | ['80%']      | False   

── L3.3 output
{
  "numeric_attestation": 0.714,
  "pass(>=0.90)": false
}



## L3.4 r_i（程式就能算）與 must-cite 清單

$$r_i=\begin{cases}1 & \text{keys}(u_i)\neq\varnothing \ \land\ \text{keys}(u_i)\subseteq K\\ 0 & \text{否則}\end{cases}$$

**加強版 recall（must-cite）**：每個主題的每個維度預先列出必引清單 $M_{t,d}$（hub 論文、對立證據），用來檢查「該出現的有沒有出現」：

$$\text{MustCiteRecall}=\frac{\sum_{t,d}|M_{t,d}\cap \text{keys}(t,d)|}{\sum_{t,d}|M_{t,d}|}$$

候選清單可以從 Step 2 的 `is_hub` 和 Step 7 的 `counter_evidence` 自動產生，再由人工確認。在 golden session 上做一次就好。

In [20]:
for u in UNITS:
    u["r"] = int(bool(u["keys"]) and set(u["keys"]) <= KEYS) if u["citable"] else None

MUST_CITE = {  # 人工確認過的必引清單（toy）
    (1, "consensus"): {"YamadaEtAl2025"},
    (1, "debates"): {"RavideshikEtAl2024", "ChenEtAl2024"},
    (2, "debates"): {"ChenEtAl2024", "SantosEtAl2022"},
}
rows, hit, total = [], 0, 0
theme_by_no = {t["number"]: t for t in SOTA["themes"]}
for (n, d), req in MUST_CITE.items():
    got = set(section_keys(theme_by_no[n], d))
    hit += len(req & got)
    total += len(req)
    rows.append((f"theme{n}.{d}", sorted(req), sorted(req - got) or "—"))
table(rows, ["段落", "必引", "缺漏"])
must_cite_recall = hit / total
table([(u["unit_id"], u["keys"], u["r"]) for u in UNITS if u["citable"]], ["citable unit", "keys", "r_i"])
show("L3.4 output", {"must_cite_recall": round(must_cite_recall, 3)})

段落               | 必引                                     | 缺漏                
-----------------+----------------------------------------+-------------------
theme1.consensus | ['YamadaEtAl2025']                     | —                 
theme1.debates   | ['ChenEtAl2024', 'RavideshikEtAl2024'] | —                 
theme2.debates   | ['ChenEtAl2024', 'SantosEtAl2022']     | ['SantosEtAl2022']

citable unit | keys                               | r_i
-------------+------------------------------------+----
u01          | ['YamadaEtAl2025', 'LiuEtAl2021a'] | 1  
u02          | ['YamadaEtAl2025']                 | 1  
u03          | ['RavideshikEtAl2024']             | 1  
u04          | ['ChenEtAl2024']                   | 1  
u05          | []                                 | 0  
u07          | ['YamadaEtAl2025']                 | 1  
u08          | ['RavideshikEtAl2024']             | 1  
u09          | ['LiuEtAl2021a']                   | 1  
u10          | ['GaddipatiLee2023', 'Park202

## L3.5 LLM-as-judge：prompt 與（mock）輸出

對每一組 (unit, key)，只把**該 key 的全文**放進 prompt，問兩個 yes/no：

1. **Support**：這篇論文本身是否主張或報告了這句（數字必須一致）？
2. **Necessary**：拿掉這個引用，這句是否就無法核對或明顯變弱？

要求模型附上 quote，方便人工抽查。下面先印出一個真實可用的 prompt，再用 `MOCK_JUDGE` 模擬模型回傳的 JSON。換成真模型時，只要把 `judge()` 改成呼叫 `llm/client`（走 cache 和 ledger；prompt 放進 `llm/prompts/` 並加版號）。

In [21]:
CITE_PROMPT = """You verify one citation in a literature review.
Claim: {claim}
Cited paper ({key}, access_level={level}):
<<<
{text}
>>>
Q1 Support: Does this paper itself state or report the claim? Numbers must match.
Q2 Necessary: If this citation were removed, would the claim become unverifiable or clearly weaker?
Answer with JSON only: {{"support": "yes|no", "support_quote": "<verbatim quote or empty>", "necessary": "yes|no"}}"""

u0 = UNITS[0]
print(CITE_PROMPT.format(claim=u0["claim"], key=u0["keys"][0], level=ACCESS[u0["keys"][0]], text=FULLTEXT[u0["keys"][0]]))

You verify one citation in a literature review.
Claim: Theta/beta neurofeedback improved global cognition in MCI.
Cited paper (YamadaEtAl2025, access_level=full-text-pdf):
<<<
Forty-eight patients (N = 48) with MCI completed 20 sessions of theta/beta neurofeedback. MoCA improved by 2.4 points relative to baseline (p = 0.003).
>>>
Q1 Support: Does this paper itself state or report the claim? Numbers must match.
Q2 Necessary: If this citation were removed, would the claim become unverifiable or clearly weaker?
Answer with JSON only: {"support": "yes|no", "support_quote": "<verbatim quote or empty>", "necessary": "yes|no"}


In [22]:
# 模擬 LLM 回傳的原始字串，依 (claim 片段, key) 索引
MOCK_JUDGE = {
    ("improved global cognition", "YamadaEtAl2025"): '{"support": "yes", "support_quote": "MoCA improved by 2.4 points", "necessary": "yes"}',
    ("improved global cognition", "LiuEtAl2021a"): '{"support": "no", "support_quote": "", "necessary": "no"}',  # 只有相關，沒有療效
    ("2.4-point MoCA gain", "YamadaEtAl2025"): '{"support": "yes", "support_quote": "MoCA improved by 2.4 points", "necessary": "yes"}',
    ("no significant MoCA difference", "RavideshikEtAl2024"): '{"support": "yes", "support_quote": "0.3 points (p = 0.61), which was not significant", "necessary": "yes"}',
    ("failed to improve memory", "ChenEtAl2024"): '{"support": "yes", "support_quote": "did not improve memory scores", "necessary": "yes"}',
    ("N = 48", "YamadaEtAl2025"): '{"support": "yes", "support_quote": "N = 48 ... 2.4 points ... p = 0.003", "necessary": "yes"}',
    ("N = 60", "RavideshikEtAl2024"): '{"support": "yes", "support_quote": "60 patients ... 0.3 points (p = 0.61)", "necessary": "yes"}',
    ("Session count correlated", "LiuEtAl2021a"): '{"support": "yes", "support_quote": "increased with session count (r = 0.29)", "necessary": "yes"}',
    ("80% accuracy", "GaddipatiLee2023"): '{"support": "no", "support_quote": "91.5% accuracy", "necessary": "yes"}',  # 數字對不上
    ("80% accuracy", "Park2022"): '{"support": "no", "support_quote": "", "necessary": "no"}',  # abstract-only
    ("baseline alpha power predicts", "ChenEtAl2024"): '{"support": "yes", "support_quote": "Baseline alpha power did not predict response", "necessary": "yes"}',
    ("baseline alpha power predicts", "LiuEtAl2021b"): '{"support": "yes", "support_quote": "weak association", "necessary": "no"}',  # 冗餘引用
    ("resting-state EEG recorded", "LiuEtAl2021a"): '{"support": "yes", "support_quote": "Resting-state EEG was recorded before training", "necessary": "yes"}',
}


def judge(claim, key):
    """Mock: 回傳 LLM 原始字串。換成真模型時改這裡。"""
    for (frag, k), raw in MOCK_JUDGE.items():
        if k == key and frag in claim:
            return raw
    raise KeyError((claim, key))


def parse_judge(raw):
    obj = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
    return obj["support"].lower() == "yes", obj["necessary"].lower() == "yes", obj.get("support_quote", "")


raw = judge(UNITS[0]["claim"], "LiuEtAl2021a")
show("mock 原始輸出", raw)
show("解析後 (support, necessary, quote)", list(parse_judge(raw)))

── mock 原始輸出
{"support": "no", "support_quote": "", "necessary": "no"}

── 解析後 (support, necessary, quote)
[
  false,
  false,
  ""
]



## L3.6 p_i（嚴格版與細緻版）

對 $r_i=1$ 的單位，令其 key 集合為 $k_1..k_m$，$s_j$ 與 $n_j$ 分別為第 $j$ 個 key 的 Support 與 Necessary 判定：

$$\text{pass}_j = s_j \land n_j \land \neg\big(\text{level}(k_j)=\text{abstract-only} \land N(u_i)\neq\varnothing\big)$$

$$p_i^{\text{strict}}=\prod_{j=1}^{m}\text{pass}_j,\qquad p_i^{\text{soft}}=\frac{1}{m}\sum_{j=1}^{m}\text{pass}_j$$

另外，數字 attestation 失敗時 $p_i=0$（Cite flow：「找不到 → 通常 $p_i=0$」）。

In [23]:
errors = []
for u in UNITS:
    if not u["citable"] or u["r"] != 1:
        continue
    passes = []
    for k in u["keys"]:
        s, n, quote = parse_judge(judge(u["claim"], k))
        abstract_num = ACCESS[k] == "abstract-only" and bool(numbers(u["claim"]))
        ok = s and n and not abstract_num
        passes.append(ok)
        if not ok:
            why = "abstract-only key supporting a number" if abstract_num else ("not supported" if not s else "not necessary")
            errors.append({"unit": u["unit_id"], "key": k, "reason": why})
    numeric_fail = "attest" in u and not u["attest"]["attested"]
    if numeric_fail:
        errors.append({"unit": u["unit_id"], "key": "*", "reason": f"numeric attestation failed: {u['attest']['missing']}"})
    u["p_strict"] = 0 if numeric_fail else int(all(passes))
    u["p_soft"] = 0.0 if numeric_fail else sum(passes) / len(passes)

table([(u["unit_id"], u["keys"], u["p_strict"], f"{u['p_soft']:.2f}") for u in UNITS if u.get("r") == 1], ["unit", "keys", "p_strict", "p_soft"])
show("errors", errors)

unit | keys                               | p_strict | p_soft
-----+------------------------------------+----------+-------
u01  | ['YamadaEtAl2025', 'LiuEtAl2021a'] | 0        | 0.50  
u02  | ['YamadaEtAl2025']                 | 1        | 1.00  
u03  | ['RavideshikEtAl2024']             | 1        | 1.00  
u04  | ['ChenEtAl2024']                   | 1        | 1.00  
u07  | ['YamadaEtAl2025']                 | 1        | 1.00  
u08  | ['RavideshikEtAl2024']             | 1        | 1.00  
u09  | ['LiuEtAl2021a']                   | 1        | 1.00  
u10  | ['GaddipatiLee2023', 'Park2022']   | 0        | 0.00  
u11  | ['ChenEtAl2024', 'LiuEtAl2021b']   | 0        | 0.50  
u12  | ['LiuEtAl2021a']                   | 1        | 1.00  

── errors
[
  {
    "unit": "u01",
    "key": "LiuEtAl2021a",
    "reason": "not supported"
  },
  {
    "unit": "u10",
    "key": "GaddipatiLee2023",
    "reason": "not supported"
  },
  {
    "unit": "u10",
    "key": "Park2022",
    "reason": "abstract

## L3.7 聚合成 Cite-p / Cite-r / F1

令 citable 單位數為 $N$，其中有引用的（$r_i=1$）單位數為 $N^{+}$：

$$\text{Cite-r}=\frac{1}{N}\sum_{i=1}^{N} r_i,\qquad \text{Cite-p}=\frac{1}{N^{+}}\sum_{i:\,r_i=1} p_i$$

$$\text{Cite-F1}=\begin{cases}\dfrac{2\cdot\text{Cite-p}\cdot\text{Cite-r}}{\text{Cite-p}+\text{Cite-r}} & \text{Cite-p}>0 \land \text{Cite-r}>0\\[4pt] 0 & \text{否則}\end{cases}$$

- 沒有引用的句子已經在 recall 扣過分，**不進 precision 的分母**，避免同一個錯誤被罰兩次。
- `integrity_fail = true` 時三個分數都是 0。

In [24]:
def cite_scores(units, integrity_fail, p_field="p_strict"):
    cit = [u for u in units if u["citable"]]
    N = len(cit)
    pos = [u for u in cit if u["r"] == 1]
    if integrity_fail:
        return {"n_claims": len(units), "n_citable": N, "integrity_fail": True, "cite_p": 0.0, "cite_r": 0.0, "cite_f1": 0.0}
    r = sum(u["r"] for u in cit) / N
    p = sum(u[p_field] for u in pos) / len(pos)
    f1 = 2 * p * r / (p + r) if p > 0 and r > 0 else 0.0
    return {"n_claims": len(units), "n_citable": N, "n_cited(N+)": len(pos), "integrity_fail": False,
            "cite_p": round(p, 3), "cite_r": round(r, 3), "cite_f1": round(f1, 3)}


integ = integrity(SOTA, KEYS)
l3 = {
    **cite_scores(UNITS, integ["integrity_fail"]),
    "cite_p_soft": cite_scores(UNITS, False, "p_soft")["cite_p"],
    "numeric_attestation": round(numeric_attestation, 3),
    "must_cite_recall": round(must_cite_recall, 3),
    "errors": errors,
}
show("L3.7 output（Cite flow 的輸出格式）", l3)
show("若 integrity_fail（幽靈鍵版本）", cite_scores(UNITS, integrity(SOTA_BAD, KEYS)["integrity_fail"]))
REPORT["L2_deterministic"] = {"integrity": integ, "structure": l2_structure,
                              "coverage": {k: (v["coverage"] if isinstance(v, dict) else v) for k, v in l2_cov.items()},
                              "numeric_attestation": round(numeric_attestation, 3), "numeric_pass": numeric_attestation >= 0.90}
REPORT["L3_cite"] = {k: v for k, v in l3.items() if k != "errors"}

── L3.7 output（Cite flow 的輸出格式）
{
  "n_claims": 13,
  "n_citable": 11,
  "n_cited(N+)": 10,
  "integrity_fail": false,
  "cite_p": 0.7,
  "cite_r": 0.909,
  "cite_f1": 0.791,
  "cite_p_soft": 0.8,
  "numeric_attestation": 0.714,
  "must_cite_recall": 0.8,
  "errors": [
    {
      "unit": "u01",
      "key": "LiuEtAl2021a",
      "reason": "not supported"
    },
    {
      "unit": "u10",
      "key": "GaddipatiLee2023",
      "reason": "not supported"
    },
    {
      "unit": "u10",
      "key": "Park2022",
      "reason": "abstract-only key supporting a number"
    },
    {
      "unit": "u10",
      "key": "*",
      "reason": "numeric attestation failed: ['80%']"
    },
    {
      "unit": "u11",
      "key": "LiuEtAl2021b",
      "reason": "not necessary"
    }
  ]
}

── 若 integrity_fail（幽靈鍵版本）
{
  "n_claims": 13,
  "n_citable": 11,
  "integrity_fail": true,
  "cite_p": 0.0,
  "cite_r": 0.0,
  "cite_f1": 0.0
}



### 延伸：Step 9 的 `\cite{}`

Step 9 的 LaTeX 用 `\cite{key}`，抽取更簡單（沿用引擎 `s9_write.CITE_RE`）。抽出 key 之後，同一套 L3 流程可以直接套用；integrity 則已由 `_validation.json` 的 `phantom_removed` 保證。

In [25]:
CITE_RE = re.compile(r"\\cite[tp]?\*?\{([^}]*)\}")
tex = r"Neurofeedback improved MoCA by 2.4 points \citep{YamadaEtAl2025}, although sham-controlled trials disagree \cite{RavideshikEtAl2024, ChenEtAl2024}."
show("Step 9 抽出的 keys", [k.strip() for m in CITE_RE.findall(tex) for k in m.split(",")])

── Step 9 抽出的 keys
[
  "YamadaEtAl2025",
  "RavideshikEtAl2024",
  "ChenEtAl2024"
]



---
# L4 Step 6 內容品質：Prometheus v2（organization / coverage / relevance）

Prometheus 採 **absolute grading**：輸入指令、待評回應、（可選的）參考答案和 score rubric，輸出 feedback 加上 `[RESULT] n`（$n\in\{1..5\}$）。

- 下面的 prompt 格式參考 prometheus-eval 的 absolute grading 模板。**正式跑時請直接用 ScholarQABench `prometheus_eval.py` 的原版模板**，才能和論文數字比較。
- 三個維度**各自打分、各自回報**，不和 Cite 或其他維度平均。
- 每個維度都要有**通用標準加上逐分 anchor**：ScholarQABench 的經驗是，rubric 有通用標準時兩位標註者的 Pearson 為 79.3，沒有時掉到 59.5。

聚合方式：同一份 SOTA 用 $S$ 次取樣（或 $S$ 個 session）時，

$$\bar s_{\text{dim}}=\frac{1}{S}\sum_{j=1}^{S}s_{\text{dim},j},\qquad \text{並回報 } \min_j s_{\text{dim},j}$$

In [26]:
RUBRICS = {
    "organization": {
        "criteria": "Is the review organised into distinct themes, each with non-overlapping Consensus / Debates / Dominant Methods / Key Results that serve their purpose, and does the cross-theme analysis connect back to the themes?",
        5: "All four dimensions present and purposeful in every theme; clear theme boundaries; cross-theme analysis builds on the themes.",
        4: "All dimensions present; occasional overlap (e.g. Consensus repeats Key Results).",
        3: "Some dimensions are perfunctory (e.g. Debates only says 'more research is needed').",
        2: "Themes read like a list of papers, organised by paper rather than by idea.",
        1: "No recognisable thematic structure.",
    },
    "coverage": {
        "criteria": "Are the full-text papers actually used, and does counter-evidence appear in Debates?",
        5: ">=80% of full-text papers cited in theme text; hubs and counter-evidence present in Debates for every theme.",
        4: ">=80% cited; counter-evidence mostly present, one pair missing.",
        3: "60-80% cited, or counter-evidence clearly missing.",
        2: "40-60% cited.",
        1: "<40% cited.",
    },
    "relevance": {
        "criteria": "Does the review stay on the PICO question?",
        5: "Tightly on PICO; no off-topic main thread.",
        4: "Occasional digressions, flagged as background.",
        3: "One theme drifts from the PICO.",
        2: "Off-topic material presented as a main thread (e.g. self-driving labs or SWE-agents in a review about something else).",
        1: "Mostly unrelated to the PICO.",
    },
}

PROMETHEUS_TEMPLATE = """###Task Description:
An instruction, a response to evaluate, and a score rubric representing an evaluation criterion are given.
1. Write a detailed feedback that assesses the quality of the response strictly based on the given score rubric, not evaluating in general.
2. After writing a feedback, write a score that is an integer between 1 and 5. You should refer to the score rubric.
3. The output format should look as follows: "(write a feedback for criteria) [RESULT] (an integer number between 1 and 5)"
4. Please do not generate any other opening, closing, and explanations.

###The instruction to evaluate:
{instruction}

###Response to evaluate:
{response}

###Score Rubrics:
[{criteria}]
Score 1: {s1}
Score 2: {s2}
Score 3: {s3}
Score 4: {s4}
Score 5: {s5}

###Feedback: """


def sota_to_text(sota):
    parts = []
    for t in sota["themes"]:
        parts.append(f"## Theme {t['number']}: {t['title_en']}")
        for d in DIMS:
            parts.append(f"### {d}\n{t[d + '_en']}")
    return "\n\n".join(parts)


def prometheus_prompt(dim, pico, sota):
    r = RUBRICS[dim]
    return PROMETHEUS_TEMPLATE.format(instruction=f"Write a thematic state-of-the-art review for PICO: {pico}", response=sota_to_text(sota),
                                      criteria=r["criteria"], s1=r[1], s2=r[2], s3=r[3], s4=r[4], s5=r[5])


PICO = "P=older adults with MCI | I=EEG neurofeedback | C=sham or no training | O=cognitive outcomes"
p = prometheus_prompt("coverage", PICO, SOTA)
print(p[:600] + "\n…\n" + p[-500:])

###Task Description:
An instruction, a response to evaluate, and a score rubric representing an evaluation criterion are given.
1. Write a detailed feedback that assesses the quality of the response strictly based on the given score rubric, not evaluating in general.
2. After writing a feedback, write a score that is an integer between 1 and 5. You should refer to the score rubric.
3. The output format should look as follows: "(write a feedback for criteria) [RESULT] (an integer number between 1 and 5)"
4. Please do not generate any other opening, closing, and explanations.

###The instruction
…
l., 2021a).

### key_results
Taken together, predictor evidence remains preliminary.

###Score Rubrics:
[Are the full-text papers actually used, and does counter-evidence appear in Debates?]
Score 1: <40% cited.
Score 2: 40-60% cited.
Score 3: 60-80% cited, or counter-evidence clearly missing.
Score 4: >=80% cited; counter-evidence mostly present, one pair missing.
Score 5: >=80% of full-text p

In [27]:
MOCK_PROMETHEUS = {  # 同一份 SOTA 取樣 3 次的模擬輸出
    "organization": ["Both themes have all four dimensions; Theme 2 Key Results is only a one-line summary. [RESULT] 4",
                     "Clear themes, but Theme 2 Key Results is thin. [RESULT] 4",
                     "Well organised; Dominant Methods in Theme 1 contains a transition sentence. [RESULT] 4"],
    "coverage": ["Two full-text papers (Okafor2023, SantosEtAl2022) are never cited, and Theme 2 Debates lacks counter-evidence. [RESULT] 2",
                 "Only about two thirds of full texts are used. [RESULT] 3",
                 "Several full-text papers are unused. [RESULT] 2"],
    "relevance": ["Stays on MCI neurofeedback throughout. [RESULT] 5",
                  "On topic. [RESULT] 5",
                  "Focused on the PICO. [RESULT] 5"],
}
RESULT_RE = re.compile(r"\[RESULT\]\s*([1-5])")

l4 = {}
for dim, outs in MOCK_PROMETHEUS.items():
    s = [int(RESULT_RE.search(o).group(1)) for o in outs]
    l4[dim] = {"samples": s, "mean": round(float(np.mean(s)), 2), "min": min(s)}
show("L4 output（三個維度分開回報）", l4)
REPORT["L4_prometheus"] = l4

── L4 output（三個維度分開回報）
{
  "organization": {
    "samples": [
      4,
      4,
      4
    ],
    "mean": 4.0,
    "min": 4
  },
  "coverage": {
    "samples": [
      2,
      3,
      2
    ],
    "mean": 2.33,
    "min": 2
  },
  "relevance": {
    "samples": [
      5,
      5,
      5
    ],
    "mean": 5.0,
    "min": 5
  }
}



---
# L5 Step 8 假說

## L5.1 ResearchAgent：15 項絕對 rubric（1–5 Likert）

ResearchAgent 把評分分成三類，每類 5 項（定義取自原論文的表）：

- **Problem**：Clarity、Relevance、Originality、Feasibility、Significance
- **Method**：Clarity、Validity、Rigorousness、Innovativeness、Generalizability
- **Experiment**：Clarity、Validity、Robustness、Feasibility、Reproducibility

Prompt 設計（依原論文的 prompt 圖）：**一次只評一個 metric**；提供 research problem 與相關論文作為 context；要求輸出 `Review / Feedback / Rating (1-5)`；`{criteria}` 放人工撰寫的 1–5 分 anchor，並寫明「避免一律給 4–5 分」。

**聚合**：每個 metric 取樣 $S$ 次後取中位數，類別分數是該類 5 項的平均：

$$m_c=\text{median}\big(s_{c,1},\dots,s_{c,S}\big),\qquad \text{Score}_{\text{type}}=\frac{1}{5}\sum_{c\in \text{type}} m_c$$

**天花板比例**（LLM 評分常集中在 4–5，這個值過高代表 rubric anchor 不夠嚴格）：

$$\text{CeilingRate}=\frac{|\{c:\ m_c\ge 4\}|}{15}$$

In [28]:
CRITERIA = {
    "Problem": {
        "Clarity": "It assesses whether the problem is defined in a clear, precise, and understandable manner.",
        "Relevance": "It measures whether the problem is pertinent and applicable to the current field or context of study.",
        "Originality": "It evaluates whether the problem presents a novel challenge or unique perspective that has not been extensively explored before.",
        "Feasibility": "It examines whether the problem can realistically be investigated or solved with the available resources and within reasonable constraints.",
        "Significance": "It assesses the importance and potential impact of solving the problem, including its contribution to the field or its broader implications.",
    },
    "Method": {
        "Clarity": "It assesses whether the method is described in a clear, precise, and understandable manner that allows for replication and comprehension of the approach.",
        "Validity": "It measures the accuracy, relevance, and soundness of the method in addressing the research problem, ensuring that it is appropriate and directly relevant to the objectives of the study.",
        "Rigorousness": "It examines the thoroughness, precision, and consistency of the method, ensuring that the approach is systematic, well-structured, and adheres to high standards of research quality.",
        "Innovativeness": "It evaluates whether the method introduces new techniques, approaches, or perspectives to the research field that differ from standard research practices and advance them in the field.",
        "Generalizability": "It assesses the extent to which the method can be applied to or is relevant for other contexts, populations, or settings beyond the scope of the study.",
    },
    "Experiment": {
        "Clarity": "It determines whether the experiment design is described in a clear, precise, and understandable manner, enabling others to grasp the setup, procedure, and expected outcomes.",
        "Validity": "It measures the appropriateness and soundness of the experimental design in accurately addressing the research questions or effectively validating the proposed methods, ensuring that the design effectively tests what it is intended to examine.",
        "Robustness": "It evaluates the durability of the experimental design across a wide range of conditions and variables, ensuring that the outcomes are not reliant on a few specific cases and remain consistent across a broad spectrum of scenarios.",
        "Feasibility": "It evaluates whether the experiment design can realistically be implemented with the available resources, time, and technological or methodological constraints, ensuring that the experiment is practical and achievable.",
        "Reproducibility": "It examines whether the information provided is sufficient and detailed enough for other researchers to reproduce the experiment using the same methodology and conditions, ensuring the reliability of the findings.",
    },
}

# step8_hypothesis.json 的 spec（toy，只列會用到的欄位）
STEP8_SPEC = {
    "gap_summary_en": "No sham-controlled trial has tested whether baseline EEG predicts neurofeedback response in MCI.",
    "research_questions": [{"id": "RQ1", "question_en": "Is a 20-session theta/beta protocol feasible for adults with MCI?"},
                           {"id": "RQ2", "question_en": "Does active neurofeedback improve MoCA more than sham?"}],
    "hypotheses": [{"id": "H1", "h0_en": "No MoCA difference between active and sham.", "h1_en": "Active NF improves MoCA by >=2 points over sham (YamadaEtAl2025).",
                    "statistical_approach_en": "Linear mixed model, group x time interaction, alpha = 0.05."}],
    "conceptual_framework_en": "Screening -> randomisation (active vs sham) -> 20 sessions -> MoCA at baseline, post, 3-month follow-up.",
    "scope_in": [{"dimension_en": "Design", "spec_en": "Double-blind RCT, 1:1, n = 60"}, {"dimension_en": "Timeframe", "spec_en": "10 weeks + 3-month follow-up"}],
    "risks": [{"risk": "Dropout", "likelihood": "Medium", "impact": "High", "mitigation": "Home-based sessions"}],
}


def spec_views(spec):
    """把 Step 8 spec 切成 ResearchAgent 的三個評分對象。"""
    problem = spec["gap_summary_en"] + "\n" + "\n".join(f"{r['id']}: {r['question_en']}" for r in spec["research_questions"])
    method = "\n".join(f"{h['id']}: H0={h['h0_en']} H1={h['h1_en']}" for h in spec["hypotheses"]) + "\nFramework: " + spec["conceptual_framework_en"]
    experiment = "\n".join(h["statistical_approach_en"] for h in spec["hypotheses"]) + "\n" + "\n".join(f"{s['dimension_en']}: {s['spec_en']}" for s in spec["scope_in"]) \
        + "\nRisks: " + "; ".join(f"{r['risk']} ({r['likelihood']}/{r['impact']}) -> {r['mitigation']}" for r in spec["risks"])
    return {"Problem": problem, "Method": method, "Experiment": experiment}


RA_SYSTEM = ("You are an AI assistant whose primary goal is to assess the quality and soundness of scientific {kind}s across diverse dimensions, "
             "in order to aid researchers in refining their {kind}s based on your evaluations and feedback, thereby enhancing the impact and reach of their work.")
RA_USER = """You are going to evaluate a scientific {kind} for its {metric}, focusing on the definition: {definition}

As part of your evaluation, you can refer to the research context below.
Research problem / gap: {problem}
Related papers (titles and abstracts): {related}

Now, proceed with your {metric} evaluation approach that should be systematic:
- Start by thoroughly reading the proposed {kind} and its rationale, keeping in mind the context provided above.
- Next, generate a review and feedback that should be constructive, helpful, and concise, focusing on the {metric} of the {kind}.
- Finally, provide a score on a 5-point Likert scale, with 1 being the lowest, please ensuring a discerning and critical evaluation to avoid a tendency towards uniformly high ratings (4-5) unless fully justified:
{criteria}

The proposed {kind}:
{content}

After your evaluation of the above content, please provide your review, feedback, and rating, in the format of
Review:
Feedback:
Rating (1-5):"""

ANCHORS_EXAMPLE = ("1 = the design cannot test the hypothesis; 2 = major confounds (no control group); "
                   "3 = control present but blinding or outcome timing unclear; 4 = sound design with minor gaps; "
                   "5 = the design directly and unambiguously tests H1 with an appropriate control and analysis.")

views = spec_views(STEP8_SPEC)
print("SYSTEM:", RA_SYSTEM.format(kind="experiment design"), "\n")
print("USER:", RA_USER.format(kind="experiment design", metric="Validity", definition=CRITERIA["Experiment"]["Validity"],
                              problem=views["Problem"], related="(shortlist abstracts …)", criteria=ANCHORS_EXAMPLE, content=views["Experiment"]))

SYSTEM: You are an AI assistant whose primary goal is to assess the quality and soundness of scientific experiment designs across diverse dimensions, in order to aid researchers in refining their experiment designs based on your evaluations and feedback, thereby enhancing the impact and reach of their work. 

USER: You are going to evaluate a scientific experiment design for its Validity, focusing on the definition: It measures the appropriateness and soundness of the experimental design in accurately addressing the research questions or effectively validating the proposed methods, ensuring that the design effectively tests what it is intended to examine.

As part of your evaluation, you can refer to the research context below.
Research problem / gap: No sham-controlled trial has tested whether baseline EEG predicts neurofeedback response in MCI.
RQ1: Is a 20-session theta/beta protocol feasible for adults with MCI?
RQ2: Does active neurofeedback improve MoCA more than sham?
Related pa

In [29]:
RATING_RE = re.compile(r"Rating\s*\(1-5\)\s*:\s*([1-5])")
# mock：每個 metric 取樣 3 次的 rating（真實情況是解析 3 份 LLM 輸出）
MOCK_RA = {
    ("Problem", "Clarity"): [4, 4, 5], ("Problem", "Relevance"): [5, 5, 5], ("Problem", "Originality"): [3, 3, 4],
    ("Problem", "Feasibility"): [4, 4, 4], ("Problem", "Significance"): [4, 3, 4],
    ("Method", "Clarity"): [4, 4, 4], ("Method", "Validity"): [4, 5, 4], ("Method", "Rigorousness"): [3, 3, 3],
    ("Method", "Innovativeness"): [2, 3, 2], ("Method", "Generalizability"): [3, 3, 3],
    ("Experiment", "Clarity"): [4, 4, 4], ("Experiment", "Validity"): [4, 4, 5], ("Experiment", "Robustness"): [3, 2, 3],
    ("Experiment", "Feasibility"): [4, 4, 4], ("Experiment", "Reproducibility"): [3, 4, 3],
}
example_output = "Review: The RCT design with sham control directly tests H1.\nFeedback: Specify the blinding check.\nRating (1-5): 4"
show("單次輸出的解析", {"raw": example_output, "rating": int(RATING_RE.search(example_output).group(1))})

med = {k: float(np.median(v)) for k, v in MOCK_RA.items()}
rows = [(t, c, MOCK_RA[(t, c)], med[(t, c)]) for (t, c) in MOCK_RA]
table(rows, ["type", "criterion", "3 samples", "median"])
l5_ra = {t: round(float(np.mean([m for (tt, _), m in med.items() if tt == t])), 2) for t in CRITERIA}
l5_ra["ceiling_rate(>=4)"] = round(sum(m >= 4 for m in med.values()) / len(med), 3)
show("L5.1 output", l5_ra)

── 單次輸出的解析
{
  "raw": "Review: The RCT design with sham control directly tests H1.\nFeedback: Specify the blinding check.\nRating (1-5): 4",
  "rating": 4
}

type       | criterion        | 3 samples | median
-----------+------------------+-----------+-------
Problem    | Clarity          | [4, 4, 5] | 4.0   
Problem    | Relevance        | [5, 5, 5] | 5.0   
Problem    | Originality      | [3, 3, 4] | 3.0   
Problem    | Feasibility      | [4, 4, 4] | 4.0   
Problem    | Significance     | [4, 3, 4] | 4.0   
Method     | Clarity          | [4, 4, 4] | 4.0   
Method     | Validity         | [4, 5, 4] | 4.0   
Method     | Rigorousness     | [3, 3, 3] | 3.0   
Method     | Innovativeness   | [2, 3, 2] | 2.0   
Method     | Generalizability | [3, 3, 3] | 3.0   
Experiment | Clarity          | [4, 4, 4] | 4.0   
Experiment | Validity         | [4, 4, 5] | 4.0   
Experiment | Robustness       | [3, 2, 3] | 3.0   
Experiment | Feasibility      | [4, 4, 4] | 4.0   
Experiment | Reproducibili

## L5.2 IdeaBench Insight Score（相對排名）

**流程**：把 target paper 的想法（用 Figure 5 的 prompt 從 abstract 萃取）和 $n$ 個生成的想法混在一起，由 judge 在不知道哪個是原作的情況下，依 quality indicator $q$（例如 novelty、feasibility）排名。

**定義**：$m$ 為 target 數；$n$ 為每題生成的想法數；$r_{\text{target}_i|q}\in\{1,\dots,n+1\}$ 為第 $i$ 個 target 的想法在排名中的名次。

$$I(\text{system},q)=\frac{1}{m}\sum_{i=1}^{m}\frac{r_{\text{target}_i|q}-1}{n}\ \in[0,1]$$

- $I=0$：target 永遠排第一，生成的想法都不如原作。
- $I=1$：target 永遠排最後，生成的想法都比原作好。
- $I>0.5$：多數生成的想法排在 target 前面。

**原論文的例子**（$n$ 必須固定，才能跨系統比較）：

| 情境 | 計算 | I |
|---|---|---|
| $n=3$，target 第 3 名 | $(3-1)/3$ | 0.667 |
| 同一個模型，$n=10$，target 第 5 名 | $(5-1)/10$ | 0.400 |
| $n=6$，target 第 4 名 → 第 5 名 | $3/6 \to 4/6$ | 0.500 → 0.667 |
| $n=10$，target 第 6 名 → 第 7 名 | $5/10 \to 6/10$ | 0.500 → 0.600 |

**位置偏誤**：IdeaBench 的模板固定把 target 放在 Hypothesis 1。若 judge 偏好第一個位置，target 的名次會被高估，I 就會被低估。建議對 $P$ 個排列取平均：

$$\bar r_{\text{target}_i}=\frac{1}{P}\sum_{\pi=1}^{P} r^{(\pi)}_{\text{target}_i}$$

In [30]:
def insight(r, n):
    return (r - 1) / n


show("原論文例子驗算", {"n=3,r=3": round(insight(3, 3), 3), "n=10,r=5": round(insight(5, 10), 3),
                          "n=6,r=4→5": [round(insight(4, 6), 3), round(insight(5, 6), 3)],
                          "n=10,r=6→7": [round(insight(6, 10), 3), round(insight(7, 10), 3)]})

# input：m=3 個 target，每題 n=3 個生成想法（3 份 Step 8 spec 各自壓縮成一段）
IDEAS = {
    "T1": {"target": "Given that theta/beta NF improves MoCA, we test it against sham in MCI.",
           "gen": ["Combine NF with transcranial stimulation to boost gains.", "Predict responders from baseline EEG microstates.", "Home-based NF with adaptive thresholds."]},
    "T2": {"target": "Hypothesis: alpha asymmetry predicts NF response.",
           "gen": ["Use closed-loop NF during sleep spindles.", "Replicate alpha asymmetry in a larger cohort.", "Test NF in a different population."]},
    "T3": {"target": "Hypothesis: NF gains persist at 6-month follow-up.",
           "gen": ["Model long-term decay of NF effects with booster sessions.", "Study NF plus cognitive training synergy.", "Assess NF cost-effectiveness."]},
}
# mock judge 的「真實品質」（novelty）；judge 另外會給第一個位置 +0.8 的偏好
HIDDEN_NOVELTY = {
    "T1": {"target": 2.0, "gen": [3.0, 2.8, 1.5]},
    "T2": {"target": 2.5, "gen": [3.2, 1.2, 1.0]},
    "T3": {"target": 1.8, "gen": [2.6, 2.4, 1.0]},
}
POSITION_BIAS = 0.8
RANK_PROMPT = "You are a reviewer tasked with ranking the quality of a set of research ideas based on their {q}. The idea with the highest {q} should be ranked first.\nPlease rank the following hypotheses:\n{hyps}"


def mock_rank_judge(order, quality):
    """order: list of idea ids in presentation order. Returns ranking text like a real LLM."""
    scored = sorted(range(len(order)), key=lambda i: -(quality[order[i]] + (POSITION_BIAS if i == 0 else 0)))
    return "\n".join(f"{r}. Hypothesis {i + 1}: (brief rationale)" for r, i in enumerate(scored, 1))


def target_rank(ranking_text, order):
    hyp_numbers = [int(h) for _, h in re.findall(r"^\s*(\d+)\.\s*Hypothesis\s*(\d+)", ranking_text, re.M)]
    return hyp_numbers.index(order.index("target") + 1) + 1


tid = "T1"
qual = {"target": HIDDEN_NOVELTY[tid]["target"], **{f"g{j}": v for j, v in enumerate(HIDDEN_NOVELTY[tid]["gen"])}}
order = ["target", "g0", "g1", "g2"]
text_map = {"target": IDEAS[tid]["target"], **{f"g{j}": g for j, g in enumerate(IDEAS[tid]["gen"])}}
print(RANK_PROMPT.format(q="novelty", hyps="\n".join(f"Hypothesis {i + 1}: {text_map[o]}" for i, o in enumerate(order))))
out = mock_rank_judge(order, qual)
print("\n── mock judge output\n" + out)
print("\n── target rank =", target_rank(out, order))

── 原論文例子驗算
{
  "n=3,r=3": 0.667,
  "n=10,r=5": 0.4,
  "n=6,r=4→5": [
    0.5,
    0.667
  ],
  "n=10,r=6→7": [
    0.5,
    0.6
  ]
}

You are a reviewer tasked with ranking the quality of a set of research ideas based on their novelty. The idea with the highest novelty should be ranked first.
Please rank the following hypotheses:
Hypothesis 1: Given that theta/beta NF improves MoCA, we test it against sham in MCI.
Hypothesis 2: Combine NF with transcranial stimulation to boost gains.
Hypothesis 3: Predict responders from baseline EEG microstates.
Hypothesis 4: Home-based NF with adaptive thresholds.

── mock judge output
1. Hypothesis 2: (brief rationale)
2. Hypothesis 1: (brief rationale)
3. Hypothesis 3: (brief rationale)
4. Hypothesis 4: (brief rationale)

── target rank = 2


In [31]:
def insight_score(hidden, n=3, randomize=True):
    per_target = {}
    for tid, h in hidden.items():
        qual = {"target": h["target"], **{f"g{j}": v for j, v in enumerate(h["gen"])}}
        ids = ["target"] + [f"g{j}" for j in range(n)]
        orders = list(itertools.permutations(ids)) if randomize else [tuple(ids)]
        ranks = [target_rank(mock_rank_judge(list(o), qual), list(o)) for o in orders]
        per_target[tid] = {"mean_rank": float(np.mean(ranks)), "I_i": insight(float(np.mean(ranks)), n)}
    I = float(np.mean([v["I_i"] for v in per_target.values()]))
    return round(I, 3), {t: {k: round(v, 3) for k, v in d.items()} for t, d in per_target.items()}


I_fixed, pt_fixed = insight_score(HIDDEN_NOVELTY, randomize=False)
I_perm, pt_perm = insight_score(HIDDEN_NOVELTY, randomize=True)
show("target 固定在 Hypothesis 1（IdeaBench 原模板）", {"I_novelty": I_fixed, "per_target": pt_fixed})
show("對全部 24 種排列取平均（建議）", {"I_novelty": I_perm, "per_target": pt_perm})

HIDDEN_FEAS = {"T1": {"target": 3.0, "gen": [1.5, 2.0, 2.8]}, "T2": {"target": 3.5, "gen": [1.0, 3.2, 2.0]}, "T3": {"target": 3.0, "gen": [1.8, 2.2, 3.4]}}
I_feas, _ = insight_score(HIDDEN_FEAS)
l5_insight = {"n": 3, "m": 3, "I_novelty(permuted)": I_perm, "I_novelty(fixed, biased)": I_fixed, "I_feasibility(permuted)": I_feas,
              "reference(IdeaBench)": "novelty 多數 > 0.6（GPT-4o high = 0.766）；feasibility 全部 < 0.5"}
show("L5.2 output", l5_insight)

── target 固定在 Hypothesis 1（IdeaBench 原模板）
{
  "I_novelty": 0.111,
  "per_target": {
    "T1": {
      "mean_rank": 2.0,
      "I_i": 0.333
    },
    "T2": {
      "mean_rank": 1.0,
      "I_i": 0.0
    },
    "T3": {
      "mean_rank": 1.0,
      "I_i": 0.0
    }
  }
}

── 對全部 24 種排列取平均（建議）
{
  "I_novelty": 0.5,
  "per_target": {
    "T1": {
      "mean_rank": 3.0,
      "I_i": 0.667
    },
    "T2": {
      "mean_rank": 1.75,
      "I_i": 0.25
    },
    "T3": {
      "mean_rank": 2.75,
      "I_i": 0.583
    }
  }
}

── L5.2 output
{
  "n": 3,
  "m": 3,
  "I_novelty(permuted)": 0.5,
  "I_novelty(fixed, biased)": 0.111,
  "I_feasibility(permuted)": 0.167,
  "reference(IdeaBench)": "novelty 多數 > 0.6（GPT-4o high = 0.766）；feasibility 全部 < 0.5"
}



## L5.3 P2 − P1：pipeline 帶來的加值

- **P1**：IdeaBench 原版，把 reference abstracts 丟給單一 prompt，直接生成 $n$ 個想法。
- **P2**：同一批 references 注入成 `step2_raw_papers.json`，跑 Steps 3–7，逐一 lock 3 個 gap，得到 3 份 Step 8 spec。

兩者用同一組 target，所以是配對比較：

$$d_i=I_i^{P2}-I_i^{P1},\qquad \bar d=\frac{1}{m}\sum_i d_i$$

95% 信賴區間用 paired bootstrap（重抽 target $B$ 次，取 $\bar d^{*}$ 的 2.5% 與 97.5% 分位數）。**區間不含 0 才算有加值**。

In [32]:
I_P1 = np.array([0.33, 0.67, 0.33, 0.67, 0.33, 0.00, 0.67, 0.33])  # 每個 target 的 I_i（toy, m=8）
I_P2 = np.array([0.67, 0.67, 0.67, 1.00, 0.33, 0.33, 0.67, 0.67])
d = I_P2 - I_P1
rng = np.random.default_rng(0)
boot = np.array([rng.choice(d, size=len(d), replace=True).mean() for _ in range(10000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
l5_p2p1 = {"I_P1": round(I_P1.mean(), 3), "I_P2": round(I_P2.mean(), 3), "mean_diff": round(d.mean(), 3),
           "95%CI": [round(lo, 3), round(hi, 3)], "significant(CI excludes 0)": bool(lo > 0 or hi < 0),
           "wins/ties/losses": [int((d > 0).sum()), int((d == 0).sum()), int((d < 0).sum())]}
show("L5.3 output", l5_p2p1)

── L5.3 output
{
  "I_P1": 0.416,
  "I_P2": 0.626,
  "mean_diff": 0.21,
  "95%CI": [
    0.084,
    0.298
  ],
  "significant(CI excludes 0)": true,
  "wins/ties/losses": [
    5,
    3,
    0
  ]
}



## L5.4 相似度輔助指標：BERTScore 與 idea overlap

用來量「pipeline 能不能重新發現真正的下一篇論文」。

**BERTScore**：生成想法 $\hat x$ 與 target abstract $x$ 的 token embedding 兩兩做 greedy matching（embedding 先正規化，所以內積 = cosine）：

$$R=\frac{1}{|x|}\sum_{x_i\in x}\max_{\hat x_j\in\hat x}x_i^{\top}\hat x_j,\qquad P=\frac{1}{|\hat x|}\sum_{\hat x_j\in\hat x}\max_{x_i\in x}x_i^{\top}\hat x_j,\qquad F_1=\frac{2PR}{P+R}$$

實務上用 `bert_score` 套件；下面用手做的 4 維向量示範計算。toy 向量全是非負值，所以分數偏高，**不能拿來和 0.718 比**，只用來說明 P、R、F1 怎麼算。IdeaBench 算出這個任務的實際上限約為 0.718（target abstract 與它自己萃取出的想法之間的分數）。

**Idea overlap**：GPT-4o 用 Figure 3 的 prompt 給 1–10 分。$n$ 個想法中，**取 BERTScore 最高的那一個**的 overlap 分數；跨 target 彙總時，論文 Table 1 回報的是 **80th percentile**。

In [33]:
EMB = {  # 手做的 4 維 token embedding（維度大致代表 protocol / intervention / outcome / control，示意用）
    "theta": [1.0, 0.1, 0.0, 0.0], "beta": [0.9, 0.2, 0.0, 0.0], "neurofeedback": [0.1, 1.0, 0.0, 0.1], "nf": [0.2, 0.9, 0.0, 0.1],
    "moca": [0.0, 0.0, 1.0, 0.0], "cognition": [0.0, 0.1, 0.7, 0.2], "sham": [0.0, 0.1, 0.0, 1.0], "improves": [0.0, 0.2, 0.4, 0.3],
}


def bertscore_toy(cand, ref):
    c = np.array([EMB[w] for w in cand.split() if w in EMB], float)
    r = np.array([EMB[w] for w in ref.split() if w in EMB], float)
    c /= np.linalg.norm(c, axis=1, keepdims=True)
    r /= np.linalg.norm(r, axis=1, keepdims=True)
    sim = r @ c.T  # |ref| x |cand|
    R, P = sim.max(axis=1).mean(), sim.max(axis=0).mean()
    return {"P": round(float(P), 3), "R": round(float(R), 3), "F1": round(float(2 * P * R / (P + R)), 3)}


ref = "theta beta neurofeedback improves moca"
cands = ["nf improves cognition", "sham neurofeedback", "theta nf moca"]
bs = {c: bertscore_toy(c, ref) for c in cands}
table([(c, v["P"], v["R"], v["F1"]) for c, v in bs.items()], ["candidate", "P", "R", "F1"])

OVERLAP_PROMPT = ("You are an expert in understanding and analyzing scientific content. Your task is to evaluate the degree of overlap between the ideas "
                  "presented in a hypothesis and the abstract of a scientific paper. ... rate the overlap on a scale of 1 to 10 ...\n"
                  "Hypothesis: {generated_research_idea}\nAbstract: {target_paper_abstract}\nRating: ...\nExplanation: ...")
MOCK_OVERLAP = {"nf improves cognition": "Rating: 6\nExplanation: same intervention, broader outcome.",
                "sham neurofeedback": "Rating: 3\nExplanation: only the control condition overlaps.",
                "theta nf moca": "Rating: 8\nExplanation: same protocol and outcome."}
best = max(cands, key=lambda c: bs[c]["F1"])
overlap_T1 = int(re.search(r"Rating:\s*(\d+)", MOCK_OVERLAP[best]).group(1))
show("T1：BERTScore 最高的想法及其 overlap", {"best_by_bertscore": best, "bertscore_F1": bs[best]["F1"], "idea_overlap": overlap_T1})

overlaps_all_targets = [overlap_T1, 5, 7, 4, 9, 6, 3, 7]  # 其餘 target 的結果（toy）
l5_sim = {"bertscore_F1_best(T1)": bs[best]["F1"], "idea_overlap_p80": round(float(np.percentile(overlaps_all_targets, 80)), 2), "bertscore_practical_ceiling": 0.718}
show("L5.4 output", l5_sim)

candidate             | P     | R     | F1   
----------------------+-------+-------+------
nf improves cognition | 0.982 | 0.736 | 0.841
sham neurofeedback    | 0.796 | 0.42  | 0.55 
theta nf moca         | 0.998 | 0.946 | 0.971

── T1：BERTScore 最高的想法及其 overlap
{
  "best_by_bertscore": "theta nf moca",
  "bertscore_F1": 0.971,
  "idea_overlap": 8
}

── L5.4 output
{
  "bertscore_F1_best(T1)": 0.971,
  "idea_overlap_p80": 7.6,
  "bertscore_practical_ceiling": 0.718
}



## L5.5 連貫性前置檢查（用本機小模型時必做）

IdeaBench 發現 Llama 3.1 8B 會產出大量不連貫的文字，而 LLM judge 會被騙，反而給出高 novelty。所以用本機 qwen 時，送進 judge 之前先過一道便宜的啟發式檢查。以下三個指標與門檻是**我的建議**，要用人工抽查來校正：

$$\text{distinct-2}=\frac{\#\text{不重複的 bigram}}{\#\text{bigram}},\qquad \text{alpha\_ratio}=\frac{\#\text{字母字元}}{\#\text{非空白字元}},\qquad \text{max\_trigram\_rep}=\max_{g}\ \text{count}(g)$$

$$\text{flag}\iff \text{distinct-2}<0.6\ \lor\ \text{alpha\_ratio}<0.7\ \lor\ \text{max\_trigram\_rep}\ge 3$$

In [34]:
def coherence(text):
    toks = text.lower().split()
    bi = list(zip(toks, toks[1:]))
    tri = Counter(zip(toks, toks[1:], toks[2:]))
    chars = [c for c in text if not c.isspace()]
    m = {"distinct_2": round(len(set(bi)) / max(1, len(bi)), 3),
         "alpha_ratio": round(sum(c.isalpha() for c in chars) / max(1, len(chars)), 3),
         "max_trigram_rep": max(tri.values()) if tri else 0}
    m["flag"] = m["distinct_2"] < 0.6 or m["alpha_ratio"] < 0.7 or m["max_trigram_rep"] >= 3
    return m


good = "We hypothesize that theta/beta neurofeedback delivered at home improves MoCA scores relative to sham in adults with MCI."
bad = "the the neurofeedback neurofeedback of the of the of the ### 1234 5678 %%% the of the of the model model"
table([(name, *coherence(t).values()) for name, t in [("good", good), ("bad", bad)]], ["text", "distinct_2", "alpha_ratio", "max_trigram_rep", "flag"])

text | distinct_2 | alpha_ratio | max_trigram_rep | flag 
-----+------------+-------------+-----------------+------
good | 1.0        | 0.981       | 1               | False
bad  | 0.65       | 0.833       | 4               | True 



## L5.6 校準 Step 7 的自評分數

Step 7 由 LLM 對每個 gap 打 novelty / feasibility（1–5），再由程式算 composite 並排序；`--auto` 模式直接 lock 第一名。拿 Step 7 的自評分數，對照外部 judge（例如 L5.2 的 Insight Score）做 Spearman 相關：

$$\rho=\text{Pearson}\big(\text{rank}(x),\ \text{rank}(y)\big)$$

同分時取平均名次。$\rho$ 很低，代表 `--auto` 選 gap 的依據不可靠，也就量化了 Checkpoint 3 需要人工把關的理由。

In [35]:
self_novelty = [4, 5, 3, 4, 2, 5, 3, 4]                            # Step 7 自評（多個 session 的 gap）
external_I = [0.67, 0.33, 0.33, 1.00, 0.00, 0.67, 0.67, 0.33]      # 外部 Insight novelty（per gap）
rho = spearman(self_novelty, external_I)
table([(i + 1, s, e, r1, r2) for i, (s, e, r1, r2) in enumerate(zip(self_novelty, external_I, rankdata(self_novelty), rankdata(external_I)))],
      ["gap", "self", "external I", "rank(self)", "rank(ext)"])
l5_cal = {"spearman_rho(self_novelty, external)": round(rho, 3)}
show("L5.6 output", l5_cal)
REPORT["L5_hypothesis"] = {"researchagent_rubric": l5_ra, "insight": l5_insight, "p2_minus_p1": l5_p2p1, "similarity": l5_sim, "step7_calibration": l5_cal}

gap | self | external I | rank(self) | rank(ext)
----+------+------------+------------+----------
1   | 4    | 0.67       | 5.0        | 6.0      
2   | 5    | 0.33       | 7.5        | 3.0      
3   | 3    | 0.33       | 2.5        | 3.0      
4   | 4    | 1.0        | 5.0        | 8.0      
5   | 2    | 0.0        | 1.0        | 1.0      
6   | 5    | 0.67       | 7.5        | 6.0      
7   | 3    | 0.67       | 2.5        | 6.0      
8   | 4    | 0.33       | 5.0        | 3.0      

── L5.6 output
{
  "spearman_rho(self_novelty, external)": 0.364
}



---
# L6 評審器本身的可信度

每一個 LLM-as-judge 指標，都要有一小份人工標註，量 judge 和人的一致程度。

**Cohen's κ**（類別標籤，例如 Support 的 yes / no）：

$$p_o=\frac{1}{N}\sum_{i}\mathbb 1[a_i=b_i],\qquad p_e=\sum_{c}\hat p_A(c)\,\hat p_B(c),\qquad \kappa=\frac{p_o-p_e}{1-p_e}$$

常用解讀（Landis & Koch）：0.41–0.60 為 moderate，0.61–0.80 為 substantial，> 0.80 為 almost perfect。

**Pearson r**（分數，例如 Prometheus 1–5 分）：

$$r=\frac{\sum_i (x_i-\bar x)(y_i-\bar y)}{\sqrt{\sum_i (x_i-\bar x)^2}\sqrt{\sum_i (y_i-\bar y)^2}}$$

**Spearman ρ**：對名次（同分取平均名次）算 Pearson。

**Kendall τ-b**（排名，例如 Step 8 的 idea ranking；可處理同分）：

$$\tau_b=\frac{n_c-n_d}{\sqrt{(n_c+n_d+t_x)(n_c+n_d+t_y)}}$$

其中 $n_c$ 為一致對數、$n_d$ 為不一致對數、$t_x$ 為只在 $x$ 同分的對數、$t_y$ 為只在 $y$ 同分的對數。

ScholarQABench 的參考值：rubric 含通用標準時，兩位標註者的 Pearson 為 79.3；不含時為 59.5。

In [36]:
# (a) Cite Support：20 組 claim-key，人工 vs judge
human = ["y", "y", "n", "y", "y", "n", "y", "y", "n", "y", "y", "y", "n", "y", "n", "y", "y", "y", "n", "y"]
judge_ = ["y", "y", "n", "y", "n", "n", "y", "y", "y", "y", "y", "y", "n", "y", "n", "y", "y", "n", "n", "y"]
kappa, p_o, p_e = cohen_kappa(human, judge_)

# (b) Prometheus coverage：10 份 SOTA，人工 vs Prometheus
human_cov = [5, 4, 2, 3, 4, 1, 5, 3, 2, 4]
prom_cov = [4, 4, 2, 3, 5, 2, 5, 2, 2, 4]

# (c) Step 8 idea ranking：一題 5 個想法，人工名次 vs judge 名次
human_rank = [1, 2, 3, 4, 5]
judge_rank = [2, 1, 3, 5, 4]

l6 = {
    "cite_support_kappa": round(kappa, 3), "p_o": round(p_o, 3), "p_e": round(p_e, 3),
    "prometheus_coverage_pearson": round(pearson(human_cov, prom_cov), 3),
    "prometheus_coverage_spearman": round(spearman(human_cov, prom_cov), 3),
    "step8_ranking_kendall_tau_b": round(kendall_tau_b(human_rank, judge_rank), 3),
}
show("L6 output", l6)

try:
    from scipy import stats
    show("scipy 交叉驗算", {
        "pearson": round(stats.pearsonr(human_cov, prom_cov)[0], 3),
        "spearman": round(stats.spearmanr(human_cov, prom_cov)[0], 3),
        "kendall_tau_b": round(stats.kendalltau(human_rank, judge_rank)[0], 3),
        "spearman(L5.6)": round(stats.spearmanr(self_novelty, external_I)[0], 3),
    })
except ImportError:
    print("scipy 未安裝，略過交叉驗算")
REPORT["L6_judge_reliability"] = l6

── L6 output
{
  "cite_support_kappa": 0.659,
  "p_o": 0.85,
  "p_e": 0.56,
  "prometheus_coverage_pearson": 0.869,
  "prometheus_coverage_spearman": 0.877,
  "step8_ranking_kendall_tau_b": 0.6
}

── scipy 交叉驗算
{
  "pearson": 0.869,
  "spearman": 0.877,
  "kendall_tau_b": 0.6,
  "spearman(L5.6)": 0.364
}



---
# 總表

各層**分區塊**回報，不混成一個平均：

- L2 是硬閘，沒過的話 L3、L4 的分數參考價值有限。
- L3（Cite）和 L4（Prometheus）各自獨立，不互相平均。
- L6 決定 L3、L4、L5 的分數可不可信。

In [37]:
gates = {
    "integrity": not REPORT["L2_deterministic"]["integrity"]["integrity_fail"],
    "structure": REPORT["L2_deterministic"]["structure"]["structure_pass"],
    "coverage(A={pdf})>=0.80": REPORT["L2_deterministic"]["coverage"]["cited 版, A={pdf}（Joanna 定義）"] >= 0.80,
    "numeric_attestation>=0.90": REPORT["L2_deterministic"]["numeric_pass"],
}
REPORT["L2_gates_summary"] = gates
print(json.dumps(REPORT, ensure_ascii=False, indent=2, default=str))

{
  "L0_reproducibility": {
    "mean_pairwise_jaccard": 0.538,
    "n_themes_values": [
      4,
      4,
      5
    ],
    "locked_gap_agreement": "2/3",
    "cite_f1_sigma_run": 0.03,
    "version_delta": 0.02,
    "delta_is_meaningful(|Δ|>2σ)": false
  },
  "L1_retrieval": {
    "specific_R@5": 0.75,
    "specific_R@20": 0.75,
    "broad_R@20": 0.5,
    "MRR": 0.411,
    "nDCG@10": 0.394,
    "step2_set_recall": 0.75,
    "step2_mean_set_size": 107.5,
    "loss_funnel": {
      "retrieved_by_provider": 1,
      "passed_year_filter": 1,
      "survived_dedup": 0,
      "passed_snowball_overlap": 0,
      "kept_in_step3": 1,
      "rank<=20": 1,
      "found": 2
    },
    "modeA_R@20": 0.688,
    "modeB_R@20": 0.562,
    "delta_step1": 0.125,
    "topic_recall": {
      "TopicRecall_step2": 0.75,
      "TopicRecall_step3": 0.417,
      "Step3_retention": 0.556,
      "|S|": 159,
      "|shortlist|": 30
    }
  },
  "L2_deterministic": {
    "integrity": {
      "n_cited_keys": 7,
 